In [1]:
import os
import sys
from pathlib import Path
import pandas as pd
import json

In [ ]:
repo_id = "Exploration-Lab/CISLR"
HF_TOKEN = os.getenv("HF_TOKEN")
if HF_TOKEN is None:
    raise ValueError("HF_TOKEN is not set")
SEED = 1990
ROOT = Path.cwd().resolve().parents[2]
sys.path.insert(0, str(ROOT))

print("Project root:", ROOT)


Project root: /Users/lokeshkumar/eMasters/EE964_Projects


In [ ]:
from src.scripts.download_from_hugging_face import download_file_from_hf

# Download the model file from Hugging Face
local_dir = ROOT / 'dataset/task3_cislr/features'
file_name = "I3D_features.pkl"
download_file_from_hf(repo_id, file_name, local_dir, repo_type="dataset", revision="main")

Total files in repo: 7
Downloaded: I3D_features.pkl
Saved to: /Users/lokeshkumar/eMasters/EE964_Projects/dataset/cislr/features/I3D_features.pkl


In [22]:
import pandas as pd
import numpy as np

pkl_file_path = ROOT / "dataset/task3_cislr/features/I3D_features.pkl"

features_df = pd.read_pickle(pkl_file_path)

print("Type:", type(features_df))
print("Shape:", getattr(features_df, "shape", None))

print("\nColumns:")
print(features_df.columns)

print("\nHead:")
print(features_df.head())
sample = features_df.iloc[0]

print("\nRow type:", type(sample))
print(sample)

feat = features_df.iloc[0]["I3D_features"]

print(type(feat))
print(np.array(feat).shape)


lengths = features_df["I3D_features"].apply(lambda x: len(x))

print("Min length:", lengths.min())
print("Max length:", lengths.max())
print("Mean length:", lengths.mean())
features_df = features_df.rename(columns={"id": "uid"})

Type: <class 'pandas.core.frame.DataFrame'>
Shape: (7050, 2)

Columns:
Index(['id', 'I3D_features'], dtype='object')

Head:
              id                                       I3D_features
0    ZcVzjZeVwj0  [[[[[0.03112409]], [[0.00099271]], [[0.000154]...
1    ZIKWlQUff9c  [[[[[0.22031154]], [[0.22767238]], [[0.1475680...
2  FnTehW6ik-Y_2  [[[[[0.20237736]], [[0.38866925]], [[0.4987630...
3    0lrX7s_3ScY  [[[[[0.04070692]], [[0.00172393]], [[0.0040934...
4    bgwSjzhyPs8  [[[[[0.]], [[0.]], [[0.]], [[0.]], [[0.0005370...

Row type: <class 'pandas.core.series.Series'>
id                                                    ZcVzjZeVwj0
I3D_features    [[[[[0.03112409]], [[0.00099271]], [[0.000154]...
Name: 0, dtype: object
<class 'numpy.ndarray'>
(1, 1024, 11, 1, 1)
Min length: 1
Max length: 1
Mean length: 1.0


In [16]:
import numpy as np

def to_td(x):
    x = np.asarray(x)
    x = np.squeeze(x)   # (1024, T)
    if x.ndim != 2:
        raise ValueError(f"Unexpected shape after squeeze: {x.shape}")
    x = x.T             # (T, 1024)
    return x

sample = to_td(features_df.iloc[0]["I3D_features"])
print("Sample final shape:", sample.shape)

lengths = features_df["I3D_features"].apply(lambda x: to_td(x).shape[0])
dims = features_df["I3D_features"].apply(lambda x: to_td(x).shape[1])

print("Min T:", lengths.min())
print("Max T:", lengths.max())
print("Mean T:", lengths.mean())
print("Unique feature dims:", dims.unique())

Sample final shape: (11, 1024)
Min T: 3
Max T: 51
Mean T: 14.406666666666666
Unique feature dims: [1024]


In [17]:
dataset_path = '/Users/lokeshkumar/eMasters/EE964_Projects/dataset/task3_cislr/splits/dataset.csv'
df_dataset = pd.read_csv(dataset_path)
print("----- Dataset CSV -----")
print(df_dataset.head())
print(type(df_dataset), getattr(df_dataset, "shape", None))
print(df_dataset.columns)

prototype_path = '/Users/lokeshkumar/eMasters/EE964_Projects/dataset/task3_cislr/splits/prototype.csv'
df_prototype = pd.read_csv(prototype_path)
print("----- Prototype CSV -----")
print(df_prototype.head())
print(type(df_prototype), getattr(df_prototype, "shape", None))
print(df_prototype.columns)

test_path = '/Users/lokeshkumar/eMasters/EE964_Projects/dataset/task3_cislr/splits/test.csv'
df_test = pd.read_csv(test_path)
print("----- Test CSV -----")
print(df_test.head())
print(type(df_test), getattr(df_test, "shape", None))
print(df_test.columns)


----- Dataset CSV -----
             uid             gloss  duration         category
0  --7-rCNOiK0_1            branch       6.0  flora and fauna
1    -02y2VaVBmw  divorce petition      11.0            legal
2    -03Dju3yPHE   air conditioner       5.0          machine
3    -0Oo1kPonYQ            stapoo       6.0            sport
4  -0XujLX2_9I_2       application       6.0           action
<class 'pandas.core.frame.DataFrame'> (7050, 4)
Index(['uid', 'gloss', 'duration', 'category'], dtype='object')
----- Prototype CSV -----
             uid             gloss  duration         category
0  --7-rCNOiK0_1            branch       6.0  flora and fauna
1    -02y2VaVBmw  divorce petition      11.0            legal
2    -03Dju3yPHE   air conditioner       5.0          machine
3    -0Oo1kPonYQ            stapoo       6.0            sport
4  -0XujLX2_9I_2       application       6.0           action
<class 'pandas.core.frame.DataFrame'> (4765, 4)
Index(['uid', 'gloss', 'duration', 'category']

In [23]:
test_uids = set(df_test["uid"])
dataset_uids = set(df_dataset["uid"])
prototype_uids = set(df_prototype["uid"])
print("UIDs in test but not in dataset:", test_uids - dataset_uids)
print("number of UIDs in test but not in prototype:", len(test_uids - prototype_uids))
I3d_uids = set(features_df["uid"])
print("UIDs in test but not in I3D features:", test_uids - I3d_uids)
print("UIDs in prototype but not in I3D features:", prototype_uids - I3d_uids)


UIDs in test but not in dataset: set()
number of UIDs in test but not in prototype: 2285
UIDs in test but not in I3D features: set()
UIDs in prototype but not in I3D features: set()


In [24]:
print(df_prototype.shape)
print(features_df.shape)
print(df_test.shape)

(4765, 4)
(7050, 2)
(2285, 4)


In [31]:
split = 0.8

train_df = df_prototype.sample(frac=split, random_state=SEED)
val_df = df_prototype.drop(train_df.index)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("Train shape:", train_df.shape)
print("Val shape:", val_df.shape)
print("Test shape:", df_test.shape)

Train shape: (3812, 4)
Val shape: (953, 4)
Test shape: (2285, 4)


In [34]:
train_df.to_csv(f"{ROOT}/dataset/task3_cislr/splits/train.csv", index=False)
val_df.to_csv(f"{ROOT}/dataset/task3_cislr/splits/val.csv", index=False)

In [4]:
from src.tasks.task3_cislr.data.dataset import Task3CISLRDataset, task3_collate_fn, reshape_i3d_feature

In [10]:
from torch.utils.data import DataLoader

train_ds = Task3CISLRDataset(
    split_csv= ROOT / "dataset/task3_cislr/splits/train.csv",
    features_pkl= ROOT / "dataset/task3_cislr/features/I3D_features.pkl",
)

print("Dataset size:", len(train_ds))
sample = train_ds[0]
print(sample["uid"], sample["gloss"], sample["features"].shape, sample["feat_len"])

loader = DataLoader(train_ds, batch_size=4, shuffle=True, collate_fn=task3_collate_fn)
batch = next(iter(loader))
print(batch["features"].shape)
print(batch["feat_len"])
print(batch["uid"][:2])

Dataset size: 3812
K1N6oh0nYvo_1 faith torch.Size([11, 1024]) 11
torch.Size([4, 25, 1024])
tensor([25,  8, 21, 13])
['fx9psR2uzxs', 'EQ1yzB0YoPY']


In [9]:
print(sample.keys())


dict_keys(['uid', 'gloss', 'features', 'feat_len'])


In [13]:
from src.tasks.task3_cislr.evaluation.baseline_i3d_nn import run_baseline

results = run_baseline(
    train_csv=ROOT / "dataset/task3_cislr/splits/train.csv",
    query_csv=ROOT / "dataset/task3_cislr/splits/val.csv",
    feature_pkl=ROOT / "dataset/task3_cislr/features/I3D_features.pkl",
)

print(results)

{'train_csv': '/Users/lokeshkumar/eMasters/EE964_Projects/dataset/task3_cislr/splits/train.csv', 'query_csv': '/Users/lokeshkumar/eMasters/EE964_Projects/dataset/task3_cislr/splits/val.csv', 'feature_pkl': '/Users/lokeshkumar/eMasters/EE964_Projects/dataset/task3_cislr/features/I3D_features.pkl', 'num_bank_samples': 3812, 'num_query_samples': 953, 'embedding_dim': 1024, 'metrics': {'top1': 0.0, 'top5': 0.0, 'top10': 0.0}}


In [14]:
from src.tasks.task3_cislr.evaluation.baseline_i3d_nn import run_baseline

results = run_baseline(
    train_csv=ROOT / "dataset/task3_cislr/splits/train.csv",
    query_csv=ROOT / "dataset/task3_cislr/splits/val.csv",
    feature_pkl=ROOT / "dataset/task3_cislr/features/I3D_features.pkl",
)

print(results)

{'train_csv': '/Users/lokeshkumar/eMasters/EE964_Projects/dataset/task3_cislr/splits/train.csv', 'query_csv': '/Users/lokeshkumar/eMasters/EE964_Projects/dataset/task3_cislr/splits/val.csv', 'feature_pkl': '/Users/lokeshkumar/eMasters/EE964_Projects/dataset/task3_cislr/features/I3D_features.pkl', 'num_bank_samples': 3812, 'num_query_samples': 953, 'embedding_dim': 1024, 'metrics': {'top1': 0.0, 'top5': 0.0, 'top10': 0.0}}


In [15]:
results = run_baseline(
    train_csv=ROOT / "dataset/task3_cislr/splits/prototype.csv",
    query_csv=ROOT / "dataset/task3_cislr/splits/test.csv",
    feature_pkl=ROOT / "dataset/task3_cislr/features/I3D_features.pkl",
)

print(results)

{'train_csv': '/Users/lokeshkumar/eMasters/EE964_Projects/dataset/task3_cislr/splits/prototype.csv', 'query_csv': '/Users/lokeshkumar/eMasters/EE964_Projects/dataset/task3_cislr/splits/test.csv', 'feature_pkl': '/Users/lokeshkumar/eMasters/EE964_Projects/dataset/task3_cislr/features/I3D_features.pkl', 'num_bank_samples': 4765, 'num_query_samples': 2285, 'embedding_dim': 1024, 'metrics': {'top1': 16.805251641137854, 'top5': 20.043763676148796, 'top10': 22.582056892778994}}


In [7]:
from src.tasks.task3_cislr.evaluation.baseline_i3d_pooling_variants import run_multi_variant_baseline
pooling_variants = ("mean", "max", "meanmax", "meanstd")
for variant in pooling_variants:
    results = run_multi_variant_baseline(
        train_csv=ROOT / "dataset/task3_cislr/splits/prototype.csv",
        query_csv=ROOT / "dataset/task3_cislr/splits/test.csv",
        feature_pkl=ROOT / "dataset/task3_cislr/features/I3D_features.pkl",
        save_path=ROOT / f"results/task3_cislr/baseline_pooling_variants_{variant}.json",
        pooling_variants=variant
    )
    print(results)



Running baseline with mean pooling...
Results saved to: /Users/lokeshkumar/eMasters/EE964_Projects/results/task3_cislr/baseline_pooling_variants_mean.json
{'train_csv': '/Users/lokeshkumar/eMasters/EE964_Projects/dataset/task3_cislr/splits/prototype.csv', 'query_csv': '/Users/lokeshkumar/eMasters/EE964_Projects/dataset/task3_cislr/splits/test.csv', 'feature_pkl': '/Users/lokeshkumar/eMasters/EE964_Projects/dataset/task3_cislr/features/I3D_features.pkl', 'device': 'mps', 'results': {'mean': {'num_bank_samples': 4765, 'num_query_samples': 2285, 'embedding_dim': 1024, 'metrics': {'top1': 16.805251641137854, 'top5': 20.043763676148796, 'top10': 22.582056892778994}}}}
Running baseline with max pooling...
Results saved to: /Users/lokeshkumar/eMasters/EE964_Projects/results/task3_cislr/baseline_pooling_variants_max.json
{'train_csv': '/Users/lokeshkumar/eMasters/EE964_Projects/dataset/task3_cislr/splits/prototype.csv', 'query_csv': '/Users/lokeshkumar/eMasters/EE964_Projects/dataset/task3_cis

In [12]:
from src.tasks.task3_cislr.models.bigru_encoder import BiGRUEncoder
from torch.utils.data import DataLoader
from src.tasks.task3_cislr.data.dataset import Task3CISLRDataset, task3_collate_fn

model = BiGRUEncoder(
    input_dim=1024,
    hidden_dim=128,
    out_dim=128,
    pooling="max",
    num_layers=1,
    dropout=0.1,
)



ds = Task3CISLRDataset(
    split_csv=ROOT / "dataset/task3_cislr/splits/prototype.csv",
    features_pkl=ROOT / "dataset/task3_cislr/features/I3D_features.pkl",
)

loader = DataLoader(ds, batch_size=4, shuffle=False, collate_fn=task3_collate_fn)
batch = next(iter(loader))


emb = model(batch["features"], batch["feat_len"])
print("Input shape:", batch["features"].shape)
print("Length shape:", batch["feat_len"].shape)
print("Embedding shape:", emb.shape)

Input shape: torch.Size([4, 31, 1024])
Length shape: torch.Size([4])
Embedding shape: torch.Size([4, 128])


In [18]:
vc = df_dataset["gloss"].value_counts()

print(vc.describe())
print("Glosses with >1 sample:", (vc > 1).sum())
print("Glosses with >2 samples:", (vc > 2).sum())
print("Glosses with >3 samples:", (vc > 3).sum())
print("Max samples for a gloss:", vc.max())

count    4764.000000
mean        1.479639
std         0.967998
min         1.000000
25%         1.000000
50%         1.000000
75%         2.000000
max        13.000000
Name: count, dtype: float64
Glosses with >1 sample: 1363
Glosses with >2 samples: 543
Glosses with >3 samples: 215
Max samples for a gloss: 13


In [4]:
import pandas as pd
from pathlib import Path

SEED = 1990

split_dir = ROOT / "dataset/task3_cislr/splits"
df_dataset = pd.read_csv(split_dir / "dataset.csv")

# 1) keep only glosses with at least 2 samples
gloss_counts = df_dataset["gloss"].value_counts()
repeated_glosses = gloss_counts[gloss_counts >= 2].index

df_repeated = df_dataset[df_dataset["gloss"].isin(repeated_glosses)].copy()

print("Original dataset shape:", df_dataset.shape)
print("Repeated-gloss subset shape:", df_repeated.shape)
print("Unique repeated glosses:", df_repeated["gloss"].nunique())
print(df_repeated["gloss"].value_counts().describe())

Original dataset shape: (7050, 4)
Repeated-gloss subset shape: (3648, 4)
Unique repeated glosses: 1363
count    1363.000000
mean        2.676449
std         1.126478
min         2.000000
25%         2.000000
50%         2.000000
75%         3.000000
max        13.000000
Name: count, dtype: float64


In [5]:
train_parts = []
val_parts = []

for gloss, group in df_repeated.groupby("gloss"):
    group = group.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    n = len(group)

    if n == 2:
        # one sample in train, one in val
        train_parts.append(group.iloc[:1])
        val_parts.append(group.iloc[1:])
    else:
        # keep at least one in val, rest in train
        val_count = max(1, round(0.2 * n))
        train_count = n - val_count

        # safety: ensure train has at least 1 sample too
        if train_count < 1:
            train_count = n - 1
            val_count = 1

        train_parts.append(group.iloc[:train_count])
        val_parts.append(group.iloc[train_count:])

train_repeated_df = pd.concat(train_parts, ignore_index=True)
val_repeated_df = pd.concat(val_parts, ignore_index=True)

print("Train repeated shape:", train_repeated_df.shape)
print("Val repeated shape:", val_repeated_df.shape)

print("Train repeated unique glosses:", train_repeated_df["gloss"].nunique())
print("Val repeated unique glosses:", val_repeated_df["gloss"].nunique())

Train repeated shape: (2275, 4)
Val repeated shape: (1373, 4)
Train repeated unique glosses: 1363
Val repeated unique glosses: 1363


In [6]:
train_glosses = set(train_repeated_df["gloss"])
val_glosses = set(val_repeated_df["gloss"])

print("Glosses in val but not in train:", len(val_glosses - train_glosses))
print("Glosses in train but not in val:", len(train_glosses - val_glosses))
print("Shared glosses:", len(train_glosses & val_glosses))

Glosses in val but not in train: 0
Glosses in train but not in val: 0
Shared glosses: 1363


In [7]:
train_uids = set(train_repeated_df["uid"])
val_uids = set(val_repeated_df["uid"])

print("UID overlap between train and val:", len(train_uids & val_uids))

UID overlap between train and val: 0


In [8]:
train_repeated_path = split_dir / "train_repeated.csv"
val_repeated_path = split_dir / "val_repeated.csv"

train_repeated_df.to_csv(train_repeated_path, index=False)
val_repeated_df.to_csv(val_repeated_path, index=False)

print("Saved:", train_repeated_path)
print("Saved:", val_repeated_path)

Saved: /Users/lokeshkumar/eMasters/EE964_Projects/dataset/task3_cislr/splits/train_repeated.csv
Saved: /Users/lokeshkumar/eMasters/EE964_Projects/dataset/task3_cislr/splits/val_repeated.csv


In [9]:
print("\nTrain gloss frequency summary:")
print(train_repeated_df["gloss"].value_counts().describe())

print("\nVal gloss frequency summary:")
print(val_repeated_df["gloss"].value_counts().describe())


Train gloss frequency summary:
count    1363.000000
mean        1.669112
std         1.083778
min         1.000000
25%         1.000000
50%         1.000000
75%         2.000000
max        10.000000
Name: count, dtype: float64

Val gloss frequency summary:
count    1363.000000
mean        1.007337
std         0.093577
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max         3.000000
Name: count, dtype: float64


In [10]:
vc = train_repeated_df["gloss"].value_counts()
print("Train glosses with >=2 samples:", (vc >= 2).sum())
print("Train glosses with >=3 samples:", (vc >= 3).sum())

Train glosses with >=2 samples: 543
Train glosses with >=3 samples: 215


In [11]:
from src.tasks.task3_cislr.data.dataset import Task3CISLRDataset, task3_collate_fn, reshape_i3d_feature
from src.tasks.task3_cislr.data.samplers import GroupedBatchSampler, build_grouped_batch_sampler

In [12]:
from torch.utils.data import DataLoader
from src.tasks.task3_cislr.data.dataset import Task3CISLRDataset, task3_collate_fn
from src.tasks.task3_cislr.data.samplers import build_grouped_batch_sampler

train_repeated_csv = ROOT / "dataset/task3_cislr/splits/train_repeated.csv"
feature_pkl = ROOT / "dataset/task3_cislr/features/I3D_features.pkl"

train_ds = Task3CISLRDataset(
    split_csv=train_repeated_csv,
    features_pkl=feature_pkl,
)

sampler = build_grouped_batch_sampler(
    split_csv=train_repeated_csv,
    n_glosses_per_batch=8,
    n_samples_per_gloss=2,
    seed=1990,
)

train_loader = DataLoader(
    train_ds,
    batch_sampler=sampler,
    collate_fn=task3_collate_fn,
)

batch = next(iter(train_loader))
print("Batch feature shape:", batch["features"].shape)
print("Batch size:", len(batch["gloss"]))
print("Glosses:", batch["gloss"])

Batch feature shape: torch.Size([16, 43, 1024])
Batch size: 16
Glosses: ['bury', 'bury', 'maha shivaratri', 'maha shivaratri', 'skin', 'skin', 'air', 'air', 'distribute', 'distribute', 'pause', 'pause', 'angle', 'angle', 'division', 'division']


In [13]:
from collections import Counter

gloss_counts = Counter(batch["gloss"])
print(gloss_counts)

Counter({'bury': 2, 'maha shivaratri': 2, 'skin': 2, 'air': 2, 'distribute': 2, 'pause': 2, 'angle': 2, 'division': 2})


In [ ]:
from src.tasks.task3_cislr.training.train_retreival import run_training

results = run_training(
    train_csv=ROOT / "dataset/task3_cislr/splits/train_repeated.csv",
    val_csv=ROOT / "dataset/task3_cislr/splits/val_repeated.csv",
    prototype_csv=ROOT / "dataset/task3_cislr/splits/prototype.csv",
    test_csv=ROOT / "dataset/task3_cislr/splits/test.csv",
    feature_pkl=ROOT / "dataset/task3_cislr/features/I3D_features.pkl",
    out_dir=ROOT / "results/task3_cislr/bigru_contrastive_sanity_h128_o128_max",
    batch_size=64,
    num_workers=0,
    hidden_dim=128,
    out_dim=128,
    pooling="max",
    num_layers=1,
    dropout=0.1,
    lr=1e-3,
    weight_decay=1e-4,
    temperature=0.07,
    epochs=5,
)

results

Epoch 01 | loss=1.9997 | val_top1=93.74 | val_top5=93.95 | val_top10=94.03
Epoch 02 | loss=1.3181 | val_top1=93.81 | val_top5=93.95 | val_top10=94.10
Epoch 03 | loss=0.8943 | val_top1=93.95 | val_top5=94.10 | val_top10=94.32
Epoch 04 | loss=0.5006 | val_top1=93.95 | val_top5=94.10 | val_top10=94.25
Epoch 05 | loss=0.3954 | val_top1=93.88 | val_top5=94.10 | val_top10=94.25

Best validation top1: 93.95484340859431
Test metrics: {'top1': 14.835886214442013, 'top5': 19.212253829321664, 'top10': 21.88183807439825}


{'train_csv': '/Users/lokeshkumar/eMasters/EE964_Projects/dataset/task3_cislr/splits/train_repeated.csv',
 'val_csv': '/Users/lokeshkumar/eMasters/EE964_Projects/dataset/task3_cislr/splits/val_repeated.csv',
 'prototype_csv': '/Users/lokeshkumar/eMasters/EE964_Projects/dataset/task3_cislr/splits/prototype.csv',
 'test_csv': '/Users/lokeshkumar/eMasters/EE964_Projects/dataset/task3_cislr/splits/test.csv',
 'feature_pkl': '/Users/lokeshkumar/eMasters/EE964_Projects/dataset/task3_cislr/features/I3D_features.pkl',
 'device': 'mps',
 'config': {'batch_size': 64,
  'hidden_dim': 128,
  'out_dim': 128,
  'pooling': 'max',
  'num_layers': 1,
  'dropout': 0.1,
  'lr': 0.001,
  'weight_decay': 0.0001,
  'temperature': 0.07,
  'epochs': 5},
 'best_val_top1': 93.95484340859431,
 'test_metrics': {'top1': 14.835886214442013,
  'top5': 19.212253829321664,
  'top10': 21.88183807439825},
 'history': [{'epoch': 1,
   'train_loss': 1.9997030562824674,
   'val_top1': 93.73634377276038,
   'val_top5': 93.9

In [5]:
from torch.utils.data import DataLoader 
from src.tasks.task3_cislr.data.dataset import Task3CISLRDataset, task3_collate_fn 
from src.tasks.task3_cislr.models.maxpool_projection import MaxPoolProjectionV1
ds = Task3CISLRDataset( split_csv=ROOT / "dataset/task3_cislr/splits/prototype.csv", features_pkl=ROOT / "dataset/task3_cislr/features/I3D_features.pkl", )
loader = DataLoader(ds, batch_size=4, shuffle=False, collate_fn=task3_collate_fn) 
batch = next(iter(loader)) 
model = MaxPoolProjectionV1( input_dim=1024, out_dim=128, dropout=0.1, ) 
emb = model(batch["features"], batch["feat_len"]) 
print("Input shape:", batch["features"].shape) 
print("Length shape:", batch["feat_len"].shape) 
print("Embedding shape:", emb.shape) 
print("Norms:", emb.norm(dim=1))

Input shape: torch.Size([4, 31, 1024])
Length shape: torch.Size([4])
Embedding shape: torch.Size([4, 128])
Norms: tensor([1.0000, 1.0000, 1.0000, 1.0000], grad_fn=<LinalgVectorNormBackward0>)


In [ ]:
from src.tasks.task3_cislr.models.maxpool_projection import MaxPoolProjectionV1
from src.scripts.get_device import get_device
device = get_device()
print("Using device:", device)
model = MaxPoolProjectionV1(
    input_dim=1024,
    out_dim=128,
    dropout=0.1,
).to(device)

Using device: mps


In [ ]:
from src.tasks.task3_cislr.training.train_retreival import run_training

results = run_training(
    train_csv=ROOT / "dataset/task3_cislr/splits/train_repeated.csv",
    val_csv=ROOT / "dataset/task3_cislr/splits/val_repeated.csv",
    prototype_csv=ROOT / "dataset/task3_cislr/splits/prototype.csv",
    test_csv=ROOT / "dataset/task3_cislr/splits/test.csv",
    feature_pkl=ROOT / "dataset/task3_cislr/features/I3D_features.pkl",
    out_dir=ROOT / "results/task3_cislr/bigru_contrastive_sanity_h128_o128_max",
    batch_size=64,
    num_workers=0,
    hidden_dim=128,
    out_dim=128,
    pooling="max",
    num_layers=1,
    dropout=0.1,
    lr=1e-3,
    weight_decay=1e-4,
    temperature=0.07,
    epochs=5,
    model=model,
)

Epoch 01 | loss=1.9077 | val_top1=26.73 | val_top5=32.92 | val_top10=36.34
Epoch 02 | loss=1.0155 | val_top1=26.80 | val_top5=32.19 | val_top10=36.20
Epoch 03 | loss=0.5579 | val_top1=27.09 | val_top5=32.77 | val_top10=36.13
Epoch 04 | loss=0.3370 | val_top1=27.31 | val_top5=33.21 | val_top10=36.85
Epoch 05 | loss=0.1811 | val_top1=27.53 | val_top5=33.21 | val_top10=37.29

Best validation top1: 27.530954115076476
Test metrics: {'top1': 16.49890590809628, 'top5': 22.27571115973742, 'top10': 25.689277899343544}


In [3]:
from torch.utils.data import DataLoader
from src.tasks.task3_cislr.data.dataset import Task3CISLRDataset, task3_collate_fn
from src.tasks.task3_cislr.models.maxpool_projection import MaxPoolProjectionV2

ds = Task3CISLRDataset(
    split_csv=ROOT / "dataset/task3_cislr/splits/prototype.csv",
    features_pkl=ROOT / "dataset/task3_cislr/features/I3D_features.pkl",
)

loader = DataLoader(ds, batch_size=4, shuffle=False, collate_fn=task3_collate_fn)
batch = next(iter(loader))

model = MaxPoolProjectionV2(
    input_dim=1024,
    hidden_dim=256,
    out_dim=128,
    dropout=0.1,
)

emb = model(batch["features"], batch["feat_len"])
print("Input shape:", batch["features"].shape)
print("Length shape:", batch["feat_len"].shape)
print("Embedding shape:", emb.shape)
print("Norms:", emb.norm(dim=1))
print(model)

Input shape: torch.Size([4, 31, 1024])
Length shape: torch.Size([4])
Embedding shape: torch.Size([4, 128])
Norms: tensor([1.0000, 1.0000, 1.0000, 1.0000], grad_fn=<LinalgVectorNormBackward0>)
MaxPoolProjectionV2(
  (fc1): Linear(in_features=1024, out_features=256, bias=True)
  (act): ReLU()
  (dropout): Dropout(p=0.1, inplace=False)
  (fc2): Linear(in_features=256, out_features=128, bias=True)
)


In [ ]:
from src.tasks.task3_cislr.training.train_retreival import run_training
from src.tasks.task3_cislr.models.maxpool_projection import MaxPoolProjectionV2

model = MaxPoolProjectionV2(
    input_dim=1024,
    hidden_dim=256,
    out_dim=128,
    dropout=0.1,
)

results = run_training(
    train_csv=ROOT / "dataset/task3_cislr/splits/train_repeated.csv",
    val_csv=ROOT / "dataset/task3_cislr/splits/val_repeated.csv",
    prototype_csv=ROOT / "dataset/task3_cislr/splits/prototype.csv",
    test_csv=ROOT / "dataset/task3_cislr/splits/test.csv",
    feature_pkl=ROOT / "dataset/task3_cislr/features/I3D_features.pkl",
    out_dir=ROOT / "results/task3_cislr/maxpool_projection_v2_h256_o128",
    batch_size=64,
    num_workers=0,
    hidden_dim=128,   # ignored by V2 model object
    out_dim=128,
    pooling="max",
    num_layers=1,     # ignored by V2 model object
    dropout=0.1,
    lr=1e-3,
    weight_decay=1e-4,
    temperature=0.07,
    epochs=5,
    model=model,
)

results["test_metrics"]

Epoch 01 | loss=2.2401 | val_top1=24.84 | val_top5=29.13 | val_top10=32.56
Epoch 02 | loss=1.6809 | val_top1=23.67 | val_top5=28.62 | val_top10=31.83
Epoch 03 | loss=1.4217 | val_top1=23.60 | val_top5=28.55 | val_top10=31.03
Epoch 04 | loss=1.1060 | val_top1=23.23 | val_top5=27.90 | val_top10=30.95
Epoch 05 | loss=1.0175 | val_top1=23.23 | val_top5=27.53 | val_top10=31.10

Best validation top1: 24.836125273124544
Test metrics: {'top1': 14.223194748358862, 'top5': 17.37417943107221, 'top10': 19.51859956236324}


{'top1': 14.223194748358862,
 'top5': 17.37417943107221,
 'top10': 19.51859956236324}

In [3]:
from src.utils.expand_grid import expand_grid

MaxPoolProjectionV2_grid = {
    "output_dim": [64, 128, 256],
    "input_dim": [1024],
    "dropout": [0.0, 0.1, 0.2],
    "lr": [1e-3, 5e-4, 2e-4],
    "weight_decay": [1e-4, 1e-5],
    "batch_size": [32],
    "temperature": [0.07, 0.1],
}

configs = expand_grid(MaxPoolProjectionV2_grid)
print(f"Total configs: {len(configs)}")
print("Sample config:", configs[0])


Total configs: 108
Sample config: {'output_dim': 64, 'input_dim': 1024, 'dropout': 0.0, 'lr': 0.001, 'weight_decay': 0.0001, 'batch_size': 32, 'temperature': 0.07}


In [3]:
import pandas as pd

from src.utils.expand_grid import expand_grid
from src.tasks.task3_cislr.training.train_retreival import run_training
from src.tasks.task3_cislr.models.maxpool_projection import MaxPoolProjectionV2

MaxPoolProjectionV2_grid = {
    "output_dim": [64, 128, 256],
    "input_dim": [1024],
    "dropout": [0.0, 0.1],
    "lr": [1e-3, 5e-4],
    "weight_decay": [1e-4],
    "batch_size": [32],
    "temperature": [0.07, 0.1],
}
NUM_EPOCHS = 5
configs = expand_grid(MaxPoolProjectionV2_grid)
print(f"Total configs: {len(configs)}")
print("Sample config:", configs[0])

all_results = []

for i, cfg in enumerate(configs, start=1):
    print("\n" + "=" * 80)
    print(f"[{i}/{len(configs)}] Running MaxPoolProjectionV2 with config:")
    print(cfg)

    exp_name = (
        f"dim{cfg['output_dim']}_"
        f"drop{str(cfg['dropout']).replace('.', 'p')}_"
        f"lr{cfg['lr']}_"
        f"wd{cfg['weight_decay']}_"
        f"temp{str(cfg['temperature']).replace('.', 'p')}"
    )

    try:
        model = MaxPoolProjectionV2(
            input_dim=cfg["input_dim"],
            hidden_dim=cfg["output_dim"] * 2,
            out_dim=cfg["output_dim"],
            dropout=cfg["dropout"],
        )

        results = run_training(
            train_csv=ROOT / "dataset/task3_cislr/splits/train_repeated.csv",
            val_csv=ROOT / "dataset/task3_cislr/splits/val_repeated.csv",
            prototype_csv=ROOT / "dataset/task3_cislr/splits/prototype.csv",
            test_csv=ROOT / "dataset/task3_cislr/splits/test.csv",
            feature_pkl=ROOT / "dataset/task3_cislr/features/I3D_features.pkl",
            out_dir=ROOT / "results/task3_cislr/maxpool_projection_v2_gridsearch" / exp_name,
            batch_size=cfg["batch_size"],
            num_workers=0,
            pooling="max",
            num_layers=1,
            dropout=cfg["dropout"],
            lr=cfg["lr"],
            weight_decay=cfg["weight_decay"],
            temperature=cfg["temperature"],
            epochs=NUM_EPOCHS,
            model=model,
        )
        
        row = {
            "exp_name": exp_name,
            **cfg,
            "best_val_top1": results["best_val_top1"],
            "val_top1": results["best_val_metrics"]["top1"],
            "val_top5": results["best_val_metrics"]["top5"],
            "val_top10": results["best_val_metrics"]["top10"],
            "test_top1": results["test_metrics"]["top1"],
            "test_top5": results["test_metrics"]["top5"],
            "test_top10": results["test_metrics"]["top10"],
        }
        all_results.append(row)

    except Exception as e:
        print(f"Failed config: {cfg}")
        print("Error:", e)
        all_results.append({
            "exp_name": exp_name,
            **cfg,
            "error": str(e),
        })

df = pd.DataFrame(all_results)
df.to_csv(ROOT / "results/task3_cislr/maxpool_projection_v2_gridsearch_results.csv", index=False)
display(df.sort_values("val_top1", ascending=False))

Total configs: 24
Sample config: {'output_dim': 64, 'input_dim': 1024, 'dropout': 0.0, 'lr': 0.001, 'weight_decay': 0.0001, 'batch_size': 32, 'temperature': 0.07}

[1/24] Running MaxPoolProjectionV2 with config:
{'output_dim': 64, 'input_dim': 1024, 'dropout': 0.0, 'lr': 0.001, 'weight_decay': 0.0001, 'batch_size': 32, 'temperature': 0.07}
Epoch 01 | loss=2.1830 | val_top1=24.33 | val_top5=29.13 | val_top10=32.41
Epoch 02 | loss=1.4008 | val_top1=24.33 | val_top5=28.19 | val_top10=30.52
Epoch 03 | loss=0.8427 | val_top1=23.38 | val_top5=27.31 | val_top10=29.79
Epoch 04 | loss=0.4689 | val_top1=23.96 | val_top5=27.75 | val_top10=30.59
Epoch 05 | loss=0.2580 | val_top1=23.38 | val_top5=27.39 | val_top10=30.66

Best validation top1: 24.326292789512017
Test metrics: {'top1': 14.61706783369803, 'top5': 17.067833698030636, 'top10': 19.168490153172865}

[2/24] Running MaxPoolProjectionV2 with config:
{'output_dim': 64, 'input_dim': 1024, 'dropout': 0.0, 'lr': 0.001, 'weight_decay': 0.0001, 'b

,exp_name,output_dim,input_dim,dropout,lr,weight_decay,batch_size,temperature,best_val_top1,val_top1,val_top5,val_top10,test_top1,test_top5,test_top10
18,dim256_drop0p0_lr0.0005_wd0.0001_temp0p07,256,1024,0.0,0.0005,0.0001,32,0.07,26.802622,26.802622,31.026948,34.668609,16.192560,19.912473,22.144420
23,dim256_drop0p1_lr0.0005_wd0.0001_temp0p1,256,1024,0.1,0.0005,0.0001,32,0.10,26.074290,26.074290,30.808449,33.648944,14.923414,19.343545,21.619256
19,dim256_drop0p0_lr0.0005_wd0.0001_temp0p1,256,1024,0.0,0.0005,0.0001,32,0.10,25.928623,25.928623,30.517116,33.576111,15.404814,18.905908,21.444201
22,dim256_drop0p1_lr0.0005_wd0.0001_temp0p07,256,1024,0.1,0.0005,0.0001,32,0.07,25.855790,25.855790,31.609614,34.085943,15.754923,19.212254,21.663020
11,dim128_drop0p0_lr0.0005_wd0.0001_temp0p1,128,1024,0.0,0.0005,0.0001,32,0.10,25.855790,25.855790,30.444283,33.284778,15.579869,19.212254,21.531729
14,dim128_drop0p1_lr0.0005_wd0.0001_temp0p07,128,1024,0.1,0.0005,0.0001,32,0.07,25.564457,25.564457,30.808449,33.940277,15.098468,18.730853,20.568928
10,dim128_drop0p0_lr0.0005_wd0.0001_temp0p07,128,1024,0.0,0.0005,0.0001,32,0.07,25.564457,25.564457,31.391114,34.304443,15.492341,18.862144,20.743982
2,dim64_drop0p0_lr0.0005_wd0.0001_temp0p07,64,1024,0.0,0.0005,0.0001,32,0.07,25.418791,25.418791,29.351784,31.900947,15.229759,18.468271,20.875274
8,dim128_drop0p0_lr0.001_wd0.0001_temp0p07,128,1024,0.0,0.0010,0.0001,32,0.07,24.908958,24.908958,29.788784,32.702112,14.617068,18.687090,20.962801
15,dim128_drop0p1_lr0.0005_wd0.0001_temp0p1,128,1024,0.1,0.0005,0.0001,32,0.10,24.908958,24.908958,28.987618,32.337946,15.054705,17.768053,19.474836


In [4]:
import pandas as pd

from src.utils.expand_grid import expand_grid
from src.tasks.task3_cislr.training.train_retreival import run_training
from src.tasks.task3_cislr.models.maxpool_projection import MaxPoolProjectionV1

In [ ]:
MaxPoolProjectionV1_grid = {
    "input_dim": [1024],
    "output_dim": [128, 256],
    "dropout": [0.0, 0.1],
    "lr": [5e-4, 2e-4],
    "weight_decay": [1e-4, 1e-5],
    "batch_size": [32],
    "temperature": [0.07, 0.1],
}


NUM_EPOCHS = 5
configs = expand_grid(MaxPoolProjectionV1_grid)
print(f"Total configs: {len(configs)}")
print("Sample config:", configs[0])

all_results = []

for i, cfg in enumerate(configs, start=1):
    print("\n" + "=" * 80)
    print(f"[{i}/{len(configs)}] Running MaxPoolProjectionV1 with config:")
    print(cfg)

    exp_name = (
        f"dim{cfg['output_dim']}_"
        f"drop{str(cfg['dropout']).replace('.', 'p')}_"
        f"lr{cfg['lr']}_"
        f"wd{cfg['weight_decay']}_"
        f"temp{str(cfg['temperature']).replace('.', 'p')}"
    )

    try:
        model = MaxPoolProjectionV1(
            input_dim=cfg["input_dim"],
            out_dim=cfg["output_dim"],
            dropout=cfg["dropout"],
        )

        results = run_training(
            train_csv=ROOT / "dataset/task3_cislr/splits/train_repeated.csv",
            val_csv=ROOT / "dataset/task3_cislr/splits/val_repeated.csv",
            prototype_csv=ROOT / "dataset/task3_cislr/splits/prototype.csv",
            test_csv=ROOT / "dataset/task3_cislr/splits/test.csv",
            feature_pkl=ROOT / "dataset/task3_cislr/features/I3D_features.pkl",
            out_dir=ROOT / "results/task3_cislr/maxpool_projection_v1_gridsearch" / exp_name,
            batch_size=cfg["batch_size"],
            num_workers=0,
            pooling="max",
            lr=cfg["lr"],
            weight_decay=cfg["weight_decay"],
            temperature=cfg["temperature"],
            epochs=NUM_EPOCHS,
            model=model,
        )
        
        row = {
            "exp_name": exp_name,
            **cfg,
            "best_val_top1": results["best_val_top1"],
            "val_top1": results["best_val_metrics"]["top1"],
            "val_top5": results["best_val_metrics"]["top5"],
            "val_top10": results["best_val_metrics"]["top10"],
            "test_top1": results["test_metrics"]["top1"],
            "test_top5": results["test_metrics"]["top5"],
            "test_top10": results["test_metrics"]["top10"],
        }
        all_results.append(row)

    except Exception as e:
        print(f"Failed config: {cfg}")
        print("Error:", e)
        all_results.append({
            "exp_name": exp_name,
            **cfg,
            "error": str(e),
        })

df = pd.DataFrame(all_results)
df.to_csv(ROOT / "results/task3_cislr/maxpool_projection_v1_gridsearch_results.csv", index=False)
display(df.sort_values("val_top1", ascending=False))

Total configs: 32
Sample config: {'input_dim': 1024, 'output_dim': 128, 'dropout': 0.0, 'lr': 0.0005, 'weight_decay': 0.0001, 'batch_size': 32, 'temperature': 0.07}

[1/32] Running MaxPoolProjectionV1 with config:
{'input_dim': 1024, 'output_dim': 128, 'dropout': 0.0, 'lr': 0.0005, 'weight_decay': 0.0001, 'batch_size': 32, 'temperature': 0.07}
Epoch 01 | loss=1.7824 | val_top1=27.39 | val_top5=33.50 | val_top10=37.00
Epoch 02 | loss=0.6985 | val_top1=27.68 | val_top5=33.43 | val_top10=36.49
Epoch 03 | loss=0.2078 | val_top1=27.24 | val_top5=33.87 | val_top10=37.22
Epoch 04 | loss=0.0385 | val_top1=27.75 | val_top5=34.16 | val_top10=37.36
Epoch 05 | loss=0.0149 | val_top1=27.46 | val_top5=33.72 | val_top10=37.65

Best validation top1: 27.749453750910416
Test metrics: {'top1': 17.505470459518598, 'top5': 23.063457330415755, 'top10': 26.170678336980306}

[2/32] Running MaxPoolProjectionV1 with config:
{'input_dim': 1024, 'output_dim': 128, 'dropout': 0.0, 'lr': 0.0005, 'weight_decay': 0.0

,exp_name,input_dim,output_dim,dropout,lr,weight_decay,batch_size,temperature,best_val_top1,val_top1,val_top5,val_top10,test_top1,test_top5,test_top10
21,dim256_drop0p0_lr0.0002_wd0.0001_temp0p1,1024,256,0.0,0.0002,0.00010,32,0.10,29.351784,29.351784,34.959942,37.873270,17.286652,21.881838,24.026258
3,dim128_drop0p0_lr0.0005_wd1e-05_temp0p1,1024,128,0.0,0.0005,0.00001,32,0.10,29.278951,29.278951,34.013110,36.853605,18.730853,24.726477,27.921225
29,dim256_drop0p1_lr0.0002_wd0.0001_temp0p1,1024,256,0.1,0.0002,0.00010,32,0.10,28.987618,28.987618,34.959942,38.601602,17.417943,22.494530,25.382932
28,dim256_drop0p1_lr0.0002_wd0.0001_temp0p07,1024,256,0.1,0.0002,0.00010,32,0.07,28.841952,28.841952,35.324108,39.184268,17.592998,21.269147,23.501094
31,dim256_drop0p1_lr0.0002_wd1e-05_temp0p1,1024,256,0.1,0.0002,0.00001,32,0.10,28.769119,28.769119,34.668609,38.674436,17.549234,21.312910,23.632385
30,dim256_drop0p1_lr0.0002_wd1e-05_temp0p07,1024,256,0.1,0.0002,0.00001,32,0.07,28.696286,28.696286,34.959942,39.329934,17.111597,21.619256,23.938731
23,dim256_drop0p0_lr0.0002_wd1e-05_temp0p1,1024,256,0.0,0.0002,0.00001,32,0.10,28.623452,28.623452,34.814275,38.383103,17.592998,22.188184,24.332604
15,dim128_drop0p1_lr0.0002_wd1e-05_temp0p1,1024,128,0.1,0.0002,0.00001,32,0.10,28.623452,28.623452,34.668609,37.363438,16.849015,21.925602,25.164114
22,dim256_drop0p0_lr0.0002_wd1e-05_temp0p07,1024,256,0.0,0.0002,0.00001,32,0.07,28.550619,28.550619,35.251275,38.455936,17.592998,22.056893,24.157549
7,dim128_drop0p0_lr0.0002_wd1e-05_temp0p1,1024,128,0.0,0.0002,0.00001,32,0.10,28.550619,28.550619,34.158776,36.999272,17.024070,20.743982,23.501094


In [6]:
import pandas as pd

train_df = pd.read_csv(ROOT / "dataset/task3_cislr/splits/train_repeated.csv")
val_df = pd.read_csv(ROOT / "dataset/task3_cislr/splits/val_repeated.csv")
test_df = pd.read_csv(ROOT / "dataset/task3_cislr/splits/test.csv")
proto_df = pd.read_csv(ROOT / "dataset/task3_cislr/splits/prototype.csv")

train_glosses = set(train_df["gloss"].unique())
val_glosses = set(val_df["gloss"].unique())
test_glosses = set(test_df["gloss"].unique())
proto_glosses = set(proto_df["gloss"].unique())

missing_val_in_proto = sorted(val_glosses - proto_glosses)
missing_test_in_proto = sorted(test_glosses - proto_glosses)

print("Unique gloss counts")
print("train:", len(train_glosses))
print("val:", len(val_glosses))
print("test:", len(test_glosses))
print("prototype:", len(proto_glosses))

print("\nMissing val glosses in prototype:", len(missing_val_in_proto))
print(missing_val_in_proto[:20])

print("\nMissing test glosses in prototype:", len(missing_test_in_proto))
print(missing_test_in_proto[:20])

Unique gloss counts
train: 1363
val: 1363
test: 1363
prototype: 4765

Missing val glosses in prototype: 0
[]

Missing test glosses in prototype: 0
[]


In [ ]:
import pandas as pd

from src.utils.expand_grid import expand_grid
from src.tasks.task3_cislr.training.train_retreival import run_training
from src.tasks.task3_cislr.models.maxpool_projection import MaxPoolProjectionV2

MaxPoolProjectionV2_grid = {
    "output_dim": [64, 128, 256],
    "input_dim": [1024],
    "dropout": [0.0, 0.1],
    "lr": [1e-3, 5e-4],
    "weight_decay": [1e-4],
    "batch_size": [32],
    "temperature": [0.07, 0.1],
}
NUM_EPOCHS = 20
configs = expand_grid(MaxPoolProjectionV2_grid)
print(f"Total configs: {len(configs)}")
print("Sample config:", configs[0])

all_results = []

for i, cfg in enumerate(configs, start=1):
    print("\n" + "=" * 80)
    print(f"[{i}/{len(configs)}] Running MaxPoolProjectionV2 with config:")
    print(cfg)

    exp_name = (
        f"mpv2_dim{cfg['output_dim']}_"
        f"drop{str(cfg['dropout']).replace('.', 'p')}_"
        f"lr{cfg['lr']}_"
        f"wd{cfg['weight_decay']}_"
        f"temp{str(cfg['temperature']).replace('.', 'p')}"
    )

    try:
        model = MaxPoolProjectionV2(
            input_dim=cfg["input_dim"],
            hidden_dim=cfg["output_dim"] * 2,
            out_dim=cfg["output_dim"],
            dropout=cfg["dropout"],
        )

        results = run_training(
            train_csv=ROOT / "dataset/task3_cislr/splits/train_repeated.csv",
            val_csv=ROOT / "dataset/task3_cislr/splits/val_repeated.csv",
            prototype_csv=ROOT / "dataset/task3_cislr/splits/prototype.csv",
            test_csv=ROOT / "dataset/task3_cislr/splits/test.csv",
            feature_pkl=ROOT / "dataset/task3_cislr/features/I3D_features.pkl",
            out_dir=ROOT / "results/task3_cislr/maxpool_projection_v2_gridsearch" / exp_name,
            batch_size=cfg["batch_size"],
            num_workers=0,
            pooling="max",
            lr=cfg["lr"],
            weight_decay=cfg["weight_decay"],
            temperature=cfg["temperature"],
            epochs=NUM_EPOCHS,
            model=model,
        )
        
        row = {
            "exp_name": exp_name,
            **cfg,
            "best_val_top1": results["best_val_top1"],
            "val_top1": results["best_val_metrics"]["top1"],
            "val_top5": results["best_val_metrics"]["top5"],
            "val_top10": results["best_val_metrics"]["top10"],
            "test_top1": results["test_metrics"]["top1"],
            "test_top5": results["test_metrics"]["top5"],
            "test_top10": results["test_metrics"]["top10"],
        }
        all_results.append(row)
        df = pd.DataFrame(all_results)
        df.to_csv(ROOT / "results/task3_cislr/maxpool_projection_v2_gridsearch_results.csv", index=False)


    except Exception as e:
        print(f"Failed config: {cfg}")
        print("Error:", e)
        all_results.append({
            "exp_name": exp_name,
            **cfg,
            "error": str(e),
        })



Total configs: 24
Sample config: {'output_dim': 64, 'input_dim': 1024, 'dropout': 0.0, 'lr': 0.001, 'weight_decay': 0.0001, 'batch_size': 32, 'temperature': 0.07}

[1/24] Running MaxPoolProjectionV2 with config:
{'output_dim': 64, 'input_dim': 1024, 'dropout': 0.0, 'lr': 0.001, 'weight_decay': 0.0001, 'batch_size': 32, 'temperature': 0.07}
Epoch 01 | loss=2.0651 | val_top1=93.88 | val_top5=94.03 | val_top10=94.03
Epoch 02 | loss=1.2754 | val_top1=93.81 | val_top5=94.10 | val_top10=94.10
Epoch 03 | loss=0.7982 | val_top1=93.88 | val_top5=94.10 | val_top10=94.10
Epoch 04 | loss=0.4974 | val_top1=93.81 | val_top5=94.03 | val_top10=94.17
Epoch 05 | loss=0.2623 | val_top1=93.88 | val_top5=94.03 | val_top10=94.25
Epoch 06 | loss=0.1366 | val_top1=93.74 | val_top5=94.10 | val_top10=94.17
Early stopping triggered at epoch 6.

Best validation top1: 93.88201019664967
Test metrics: {'top1': 14.00437636761488, 'top5': 17.855579868708972, 'top10': 19.474835886214443}

[2/24] Running MaxPoolProjecti

,exp_name,output_dim,input_dim,dropout,lr,weight_decay,batch_size,temperature,best_val_top1,val_top1,val_top5,val_top10,test_top1,test_top5,test_top10
2,mpv2_dim64_drop0p0_lr0.0005_wd0.0001_temp0p07,64,1024,0.0,0.0005,0.0001,32,0.07,94.100510,94.100510,94.173343,94.246176,14.967177,18.687090,20.787746
18,mpv2_dim256_drop0p0_lr0.0005_wd0.0001_temp0p07,256,1024,0.0,0.0005,0.0001,32,0.07,94.100510,94.100510,94.246176,94.391843,16.455142,21.444201,23.938731
10,mpv2_dim128_drop0p0_lr0.0005_wd0.0001_temp0p07,128,1024,0.0,0.0005,0.0001,32,0.07,94.100510,94.100510,94.391843,94.391843,16.717724,21.531729,24.595186
23,mpv2_dim256_drop0p1_lr0.0005_wd0.0001_temp0p1,256,1024,0.1,0.0005,0.0001,32,0.10,94.027677,94.027677,94.100510,94.173343,15.317287,20.043764,22.144420
22,mpv2_dim256_drop0p1_lr0.0005_wd0.0001_temp0p07,256,1024,0.1,0.0005,0.0001,32,0.07,94.027677,94.027677,94.100510,94.319009,15.142232,19.168490,21.181619
20,mpv2_dim256_drop0p1_lr0.001_wd0.0001_temp0p07,256,1024,0.1,0.0010,0.0001,32,0.07,94.027677,94.027677,94.173343,94.173343,13.785558,18.074398,19.693654
15,mpv2_dim128_drop0p1_lr0.0005_wd0.0001_temp0p1,128,1024,0.1,0.0005,0.0001,32,0.10,94.027677,94.027677,94.027677,94.173343,15.010941,18.555799,20.612691
16,mpv2_dim256_drop0p0_lr0.001_wd0.0001_temp0p07,256,1024,0.0,0.0010,0.0001,32,0.07,93.954843,93.954843,94.027677,94.100510,13.960613,18.118162,20.437637
3,mpv2_dim64_drop0p0_lr0.0005_wd0.0001_temp0p1,64,1024,0.0,0.0005,0.0001,32,0.10,93.954843,93.954843,94.027677,94.319009,15.098468,19.387309,22.275711
21,mpv2_dim256_drop0p1_lr0.001_wd0.0001_temp0p1,256,1024,0.1,0.0010,0.0001,32,0.10,93.954843,93.954843,94.173343,94.319009,14.135667,16.761488,19.124726


In [4]:
MaxPoolProjectionV1_grid = {
    "input_dim": [1024],
    "output_dim": [128, 256],
    "dropout": [0.0, 0.1],
    "lr": [5e-4, 2e-4],
    "weight_decay": [1e-4, 1e-5],
    "batch_size": [32],
    "temperature": [0.07, 0.1],
}


NUM_EPOCHS = 20
configs = expand_grid(MaxPoolProjectionV1_grid)
print(f"Total configs: {len(configs)}")
print("Sample config:", configs[0])

all_results = []

for i, cfg in enumerate(configs, start=1):
    print("\n" + "=" * 80)
    print(f"[{i}/{len(configs)}] Running MaxPoolProjectionV1 with config:")
    print(cfg)

    exp_name = (
        f"mpv1_dim{cfg['output_dim']}_"
        f"drop{str(cfg['dropout']).replace('.', 'p')}_"
        f"lr{cfg['lr']}_"
        f"wd{cfg['weight_decay']}_"
        f"temp{str(cfg['temperature']).replace('.', 'p')}"
    )

    try:
        model = MaxPoolProjectionV1(
            input_dim=cfg["input_dim"],
            out_dim=cfg["output_dim"],
            dropout=cfg["dropout"],
        )

        results = run_training(
            train_csv=ROOT / "dataset/task3_cislr/splits/train_repeated.csv",
            val_csv=ROOT / "dataset/task3_cislr/splits/val_repeated.csv",
            prototype_csv=ROOT / "dataset/task3_cislr/splits/prototype.csv",
            test_csv=ROOT / "dataset/task3_cislr/splits/test.csv",
            feature_pkl=ROOT / "dataset/task3_cislr/features/I3D_features.pkl",
            out_dir=ROOT / "results/task3_cislr/maxpool_projection_v1_gridsearch" / exp_name,
            batch_size=cfg["batch_size"],
            num_workers=0,
            pooling="max",
            lr=cfg["lr"],
            weight_decay=cfg["weight_decay"],
            temperature=cfg["temperature"],
            epochs=NUM_EPOCHS,
            model=model,
        )
        
        row = {
            "exp_name": exp_name,
            **cfg,
            "best_val_top1": results["best_val_top1"],
            "val_top1": results["best_val_metrics"]["top1"],
            "val_top5": results["best_val_metrics"]["top5"],
            "val_top10": results["best_val_metrics"]["top10"],
            "test_top1": results["test_metrics"]["top1"],
            "test_top5": results["test_metrics"]["top5"],
            "test_top10": results["test_metrics"]["top10"],
        }
        all_results.append(row)
        df = pd.DataFrame(all_results)
        df.to_csv(ROOT / "results/task3_cislr/maxpool_projection_v1_gridsearch_results.csv", index=False)

    except Exception as e:
        print(f"Failed config: {cfg}")
        print("Error:", e)
        all_results.append({
            "exp_name": exp_name,
            **cfg,
            "error": str(e),
        })

display(df.sort_values("val_top1", ascending=False))

Total configs: 32
Sample config: {'input_dim': 1024, 'output_dim': 128, 'dropout': 0.0, 'lr': 0.0005, 'weight_decay': 0.0001, 'batch_size': 32, 'temperature': 0.07}

[1/32] Running MaxPoolProjectionV1 with config:
{'input_dim': 1024, 'output_dim': 128, 'dropout': 0.0, 'lr': 0.0005, 'weight_decay': 0.0001, 'batch_size': 32, 'temperature': 0.07}
Failed config: {'input_dim': 1024, 'output_dim': 128, 'dropout': 0.0, 'lr': 0.0005, 'weight_decay': 0.0001, 'batch_size': 32, 'temperature': 0.07}
Error: name 'MaxPoolProjectionV1' is not defined

[2/32] Running MaxPoolProjectionV1 with config:
{'input_dim': 1024, 'output_dim': 128, 'dropout': 0.0, 'lr': 0.0005, 'weight_decay': 0.0001, 'batch_size': 32, 'temperature': 0.1}
Failed config: {'input_dim': 1024, 'output_dim': 128, 'dropout': 0.0, 'lr': 0.0005, 'weight_decay': 0.0001, 'batch_size': 32, 'temperature': 0.1}
Error: name 'MaxPoolProjectionV1' is not defined

[3/32] Running MaxPoolProjectionV1 with config:
{'input_dim': 1024, 'output_dim': 

,exp_name,output_dim,input_dim,dropout,lr,weight_decay,batch_size,temperature,best_val_top1,val_top1,val_top5,val_top10,test_top1,test_top5,test_top10
2,mpv2_dim64_drop0p0_lr0.0005_wd0.0001_temp0p07,64,1024,0.0,0.0005,0.0001,32,0.07,94.100510,94.100510,94.173343,94.246176,14.967177,18.687090,20.787746
18,mpv2_dim256_drop0p0_lr0.0005_wd0.0001_temp0p07,256,1024,0.0,0.0005,0.0001,32,0.07,94.100510,94.100510,94.246176,94.391843,16.455142,21.444201,23.938731
10,mpv2_dim128_drop0p0_lr0.0005_wd0.0001_temp0p07,128,1024,0.0,0.0005,0.0001,32,0.07,94.100510,94.100510,94.391843,94.391843,16.717724,21.531729,24.595186
23,mpv2_dim256_drop0p1_lr0.0005_wd0.0001_temp0p1,256,1024,0.1,0.0005,0.0001,32,0.10,94.027677,94.027677,94.100510,94.173343,15.317287,20.043764,22.144420
22,mpv2_dim256_drop0p1_lr0.0005_wd0.0001_temp0p07,256,1024,0.1,0.0005,0.0001,32,0.07,94.027677,94.027677,94.100510,94.319009,15.142232,19.168490,21.181619
20,mpv2_dim256_drop0p1_lr0.001_wd0.0001_temp0p07,256,1024,0.1,0.0010,0.0001,32,0.07,94.027677,94.027677,94.173343,94.173343,13.785558,18.074398,19.693654
15,mpv2_dim128_drop0p1_lr0.0005_wd0.0001_temp0p1,128,1024,0.1,0.0005,0.0001,32,0.10,94.027677,94.027677,94.027677,94.173343,15.010941,18.555799,20.612691
16,mpv2_dim256_drop0p0_lr0.001_wd0.0001_temp0p07,256,1024,0.0,0.0010,0.0001,32,0.07,93.954843,93.954843,94.027677,94.100510,13.960613,18.118162,20.437637
3,mpv2_dim64_drop0p0_lr0.0005_wd0.0001_temp0p1,64,1024,0.0,0.0005,0.0001,32,0.10,93.954843,93.954843,94.027677,94.319009,15.098468,19.387309,22.275711
21,mpv2_dim256_drop0p1_lr0.001_wd0.0001_temp0p1,256,1024,0.1,0.0010,0.0001,32,0.10,93.954843,93.954843,94.173343,94.319009,14.135667,16.761488,19.124726


In [6]:
df_mp_v2 = pd.read_csv(ROOT / "results/task3_cislr/maxpool_projection_v2_gridsearch_results.csv")
display(df_mp_v2.sort_values("test_top1", ascending=False))

,exp_name,output_dim,input_dim,dropout,lr,weight_decay,batch_size,temperature,best_val_top1,val_top1,val_top5,val_top10,test_top1,test_top5,test_top10
10,mpv2_dim128_drop0p0_lr0.0005_wd0.0001_temp0p07,128,1024,0.0,0.0005,0.0001,32,0.07,94.100510,94.100510,94.391843,94.391843,16.717724,21.531729,24.595186
18,mpv2_dim256_drop0p0_lr0.0005_wd0.0001_temp0p07,256,1024,0.0,0.0005,0.0001,32,0.07,94.100510,94.100510,94.246176,94.391843,16.455142,21.444201,23.938731
19,mpv2_dim256_drop0p0_lr0.0005_wd0.0001_temp0p1,256,1024,0.0,0.0005,0.0001,32,0.10,93.954843,93.954843,93.954843,94.100510,15.711160,19.299781,21.925602
11,mpv2_dim128_drop0p0_lr0.0005_wd0.0001_temp0p1,128,1024,0.0,0.0005,0.0001,32,0.10,93.954843,93.954843,94.027677,94.246176,15.317287,19.080963,21.269147
23,mpv2_dim256_drop0p1_lr0.0005_wd0.0001_temp0p1,256,1024,0.1,0.0005,0.0001,32,0.10,94.027677,94.027677,94.100510,94.173343,15.317287,20.043764,22.144420
22,mpv2_dim256_drop0p1_lr0.0005_wd0.0001_temp0p07,256,1024,0.1,0.0005,0.0001,32,0.07,94.027677,94.027677,94.100510,94.319009,15.142232,19.168490,21.181619
3,mpv2_dim64_drop0p0_lr0.0005_wd0.0001_temp0p1,64,1024,0.0,0.0005,0.0001,32,0.10,93.954843,93.954843,94.027677,94.319009,15.098468,19.387309,22.275711
15,mpv2_dim128_drop0p1_lr0.0005_wd0.0001_temp0p1,128,1024,0.1,0.0005,0.0001,32,0.10,94.027677,94.027677,94.027677,94.173343,15.010941,18.555799,20.612691
2,mpv2_dim64_drop0p0_lr0.0005_wd0.0001_temp0p07,64,1024,0.0,0.0005,0.0001,32,0.07,94.100510,94.100510,94.173343,94.246176,14.967177,18.687090,20.787746
14,mpv2_dim128_drop0p1_lr0.0005_wd0.0001_temp0p07,128,1024,0.1,0.0005,0.0001,32,0.07,93.954843,93.954843,94.027677,94.100510,14.748359,18.161926,20.437637


In [7]:
df_mp_v1 = pd.read_csv(ROOT / "results/task3_cislr/maxpool_projection_v1_gridsearch_results.csv")
display(df_mp_v1.sort_values("test_top1", ascending=False))

,exp_name,input_dim,output_dim,dropout,lr,weight_decay,batch_size,temperature,best_val_top1,val_top1,val_top5,val_top10,test_top1,test_top5,test_top10
17,dim256_drop0p0_lr0.0005_wd0.0001_temp0p1,1024,256,0.0,0.0005,0.00010,32,0.10,28.477786,28.477786,33.940277,37.217771,18.905908,25.426696,28.577681
3,dim128_drop0p0_lr0.0005_wd1e-05_temp0p1,1024,128,0.0,0.0005,0.00001,32,0.10,29.278951,29.278951,34.013110,36.853605,18.730853,24.726477,27.921225
1,dim128_drop0p0_lr0.0005_wd0.0001_temp0p1,1024,128,0.0,0.0005,0.00010,32,0.10,27.458121,27.458121,32.993445,36.270940,18.555799,24.507659,27.264770
2,dim128_drop0p0_lr0.0005_wd1e-05_temp0p07,1024,128,0.0,0.0005,0.00001,32,0.07,28.186453,28.186453,34.595776,37.363438,18.205689,23.457330,27.483589
18,dim256_drop0p0_lr0.0005_wd1e-05_temp0p07,1024,256,0.0,0.0005,0.00001,32,0.07,27.895120,27.895120,34.450109,38.091770,17.986871,23.238512,26.870897
28,dim256_drop0p1_lr0.0002_wd0.0001_temp0p07,1024,256,0.1,0.0002,0.00010,32,0.07,28.841952,28.841952,35.324108,39.184268,17.592998,21.269147,23.501094
23,dim256_drop0p0_lr0.0002_wd1e-05_temp0p1,1024,256,0.0,0.0002,0.00001,32,0.10,28.623452,28.623452,34.814275,38.383103,17.592998,22.188184,24.332604
22,dim256_drop0p0_lr0.0002_wd1e-05_temp0p07,1024,256,0.0,0.0002,0.00001,32,0.07,28.550619,28.550619,35.251275,38.455936,17.592998,22.056893,24.157549
31,dim256_drop0p1_lr0.0002_wd1e-05_temp0p1,1024,256,0.1,0.0002,0.00001,32,0.10,28.769119,28.769119,34.668609,38.674436,17.549234,21.312910,23.632385
26,dim256_drop0p1_lr0.0005_wd1e-05_temp0p07,1024,256,0.1,0.0005,0.00001,32,0.07,27.967953,27.967953,34.085943,36.999272,17.549234,23.150985,26.345733


In [8]:
import json

path = ROOT / "results/task3_cislr/maxpool_projection_v2_gridsearch" / \
    "mpv2_dim128_drop0p0_lr0.0005_wd0.0001_temp0p07" / \
    "results_MaxPoolProjectionV2_h256_o128_lr0.0005_temp0.07.json"

with open(path, "r") as f:
    obj = json.load(f)

print(obj["best_val_top1"])
print(obj["best_val_metrics"])
print(obj["test_metrics"])
print(obj["history"][:3])
print(obj["history"][-3:])

94.10050983248361
{'top1': 94.10050983248361, 'top5': 94.3918426802622, 'top10': 94.3918426802622}
{'top1': 16.717724288840262, 'top5': 21.53172866520788, 'top10': 24.595185995623634}
[{'epoch': 1, 'train_loss': 1.9119267834557427, 'val_top1': 93.88201019664967, 'val_top5': 94.02767662053897, 'val_top10': 94.10050983248361}, {'epoch': 2, 'train_loss': 1.0314695756468508, 'val_top1': 93.88201019664967, 'val_top5': 94.17334304442826, 'val_top10': 94.31900946831755}, {'epoch': 3, 'train_loss': 0.48663332093920975, 'val_top1': 93.95484340859431, 'val_top5': 94.02767662053897, 'val_top10': 94.2461762563729}]
[{'epoch': 9, 'train_loss': 0.004803919657650921, 'val_top1': 94.10050983248361, 'val_top5': 94.3918426802622, 'val_top10': 94.3918426802622}, {'epoch': 10, 'train_loss': 0.004208534924934308, 'val_top1': 94.10050983248361, 'val_top5': 94.3918426802622, 'val_top10': 94.3918426802622}, {'epoch': 11, 'train_loss': 0.0037619197223749427, 'val_top1': 94.02767662053897, 'val_top5': 94.391842

In [10]:
from src.tasks.task3_cislr.training.train_retreival import make_loader, evaluate_retrieval
from src.tasks.task3_cislr.models.maxpool_projection import MaxPoolProjectionV2
import torch
from src.scripts.get_device import get_device


device = get_device()

model = MaxPoolProjectionV2(
    input_dim=1024,
    hidden_dim=256,
    out_dim=128,
    dropout=0.0,
).to(device)

ckpt_path = ROOT / "results/task3_cislr/maxpool_projection_v2_gridsearch" / \
    "mpv2_dim128_drop0p0_lr0.0005_wd0.0001_temp0p07" / \
    "MaxPoolProjectionV2_h256_o128_lr0.0005_temp0.07_best.pt"

ckpt = torch.load(ckpt_path, map_location=device)
model.load_state_dict(ckpt["model_state_dict"])

prototype_loader = make_loader(
    split_csv=ROOT / "dataset/task3_cislr/splits/prototype.csv",
    feature_pkl=ROOT / "dataset/task3_cislr/features/I3D_features.pkl",
    batch_size=32,
    shuffle=False,
    num_workers=0,
)

val_loader = make_loader(
    split_csv=ROOT / "dataset/task3_cislr/splits/val_repeated.csv",
    feature_pkl=ROOT / "dataset/task3_cislr/features/I3D_features.pkl",
    batch_size=32,
    shuffle=False,
    num_workers=0,
)

test_loader = make_loader(
    split_csv=ROOT / "dataset/task3_cislr/splits/test.csv",
    feature_pkl=ROOT / "dataset/task3_cislr/features/I3D_features.pkl",
    batch_size=32,
    shuffle=False,
    num_workers=0,
)

val_metrics = evaluate_retrieval(model, prototype_loader, val_loader, device)
test_metrics = evaluate_retrieval(model, prototype_loader, test_loader, device)

print("Manual val:", val_metrics)
print("Manual test:", test_metrics)

Manual val: {'top1': 94.10050983248361, 'top5': 94.3918426802622, 'top10': 94.3918426802622}
Manual test: {'top1': 16.717724288840262, 'top5': 21.53172866520788, 'top10': 24.595185995623634}


In [11]:
import pandas as pd

val_df = pd.read_csv(ROOT / "dataset/task3_cislr/splits/val_repeated.csv")
proto_df = pd.read_csv(ROOT / "dataset/task3_cislr/splits/prototype.csv")

val_ids = set(val_df["uid"]) if "uid" in val_df.columns else set(val_df.iloc[:, 0])
proto_ids = set(proto_df["uid"]) if "uid" in proto_df.columns else set(proto_df.iloc[:, 0])

overlap = val_ids & proto_ids
print("Exact ID overlap:", len(overlap))
print(list(sorted(overlap))[:20])

Exact ID overlap: 1281
['--7-rCNOiK0_1', '-0XujLX2_9I_2', '-3db5Hhn4mI_1', '-5p4hn8J87A_2', '-65ryXubYjo', '-8CjLUfRlbw_1', '-A1P5WbuNo8', '-DDBcacIfeA_1', '-DHtnmdlaGk', '-Em79Yvwac4_1', '-H_VzAnfyKk', '-It9bwqowig', '-JcpfOHc0nE', '-Jr9EZz7Epo_1', '-NIz596Y27Q', '-NpA8OS_Oj0_1', '-Oq_e89164c_1', '-PGXFLhDfS0_1', '-R02rqkyxHc_1', '-SVSp2VtfRw_1']


In [13]:
import re

def get_base_id(uid: str) -> str:
    return re.sub(r'_(w|d|e\d+)$', '', str(uid))

val_base = set(val_df["uid"].map(get_base_id))
proto_base = set(proto_df["uid"].map(get_base_id))

base_overlap = val_base & proto_base
print("Base ID overlap:", len(base_overlap))
print(list(sorted(base_overlap))[:20])

Base ID overlap: 1281
['--7-rCNOiK0_1', '-0XujLX2_9I_2', '-3db5Hhn4mI_1', '-5p4hn8J87A_2', '-65ryXubYjo', '-8CjLUfRlbw_1', '-A1P5WbuNo8', '-DDBcacIfeA_1', '-DHtnmdlaGk', '-Em79Yvwac4_1', '-H_VzAnfyKk', '-It9bwqowig', '-JcpfOHc0nE', '-Jr9EZz7Epo_1', '-NIz596Y27Q', '-NpA8OS_Oj0_1', '-Oq_e89164c_1', '-PGXFLhDfS0_1', '-R02rqkyxHc_1', '-SVSp2VtfRw_1']


In [14]:
print("Val rows:", len(val_df))
print("Prototype rows:", len(proto_df))
print("Val unique gloss:", val_df["gloss"].nunique())
print("Prototype unique gloss:", proto_df["gloss"].nunique())

proto_counts = proto_df["gloss"].value_counts()
print(proto_counts.describe())

Val rows: 1373
Prototype rows: 4765
Val unique gloss: 1363
Prototype unique gloss: 4764
count    4764.0
mean        1.0
std         0.0
min         1.0
25%         1.0
50%         1.0
75%         1.0
max         1.0
Name: count, dtype: float64


In [15]:
df_val_repeated = pd.read_csv(ROOT / "dataset/task3_cislr/splits/val_repeated.csv")
df_prototype = pd.read_csv(ROOT / "dataset/task3_cislr/splits/prototype.csv")  
print(df_val_repeated.columns)
print(df_prototype.columns)

Index(['uid', 'gloss', 'duration', 'category'], dtype='object')
Index(['uid', 'gloss', 'duration', 'category'], dtype='object')


In [16]:
import pandas as pd

df_val = pd.read_csv(ROOT / "dataset/task3_cislr/splits/val_repeated.csv")
df_proto = pd.read_csv(ROOT / "dataset/task3_cislr/splits/prototype.csv")

proto_uids = set(df_proto["uid"])
df_val_clean = df_val[~df_val["uid"].isin(proto_uids)].copy()

print("Old val rows:", len(df_val))
print("Clean val rows:", len(df_val_clean))
print("Removed rows:", len(df_val) - len(df_val_clean))

df_val_clean.to_csv(ROOT / "dataset/task3_cislr/splits/val_repeated_no_proto_overlap.csv", index=False)

Old val rows: 1373
Clean val rows: 92
Removed rows: 1281


In [18]:
BiGRUPoolEncoder_grid = {
    "input_dim": [1024],
    "hidden_dim": [128, 256, 512],
    "output_dim": [128, 256, 512],
    "dropout": [0.0, 0.1, 0.2],
    "lr": [1e-5, 5e-4, 2e-4],
    "weight_decay": [1e-4, 1e-5],
    "batch_size": [32],
    "temperature": [0.07, 0.1],
    "num_layers": [1, 2],
    "pool": ["max", "mean"],
}

NUM_EPOCHS = 20
configs = expand_grid(BiGRUPoolEncoder_grid)
print(f"Total configs: {len(configs)}")
print("Sample config:", configs[0])

Total configs: 1296
Sample config: {'input_dim': 1024, 'hidden_dim': 128, 'output_dim': 128, 'dropout': 0.0, 'lr': 1e-05, 'weight_decay': 0.0001, 'batch_size': 32, 'temperature': 0.07, 'num_layers': 1, 'pool': 'max'}


In [21]:
from src.tasks.task3_cislr.models.bigru_encoder import BiGRUPoolEncoder, BiGRUAttentionEncoder

BiGRUAttentionEncoder_grid_stage1 = {
    "input_dim": [1024],
    "hidden_dim": [256],
    "output_dim": [256],
    "dropout": [0.0, 0.1],
    "lr": [5e-4, 2e-4],
    "weight_decay": [1e-4],
    "batch_size": [32],
    "temperature": [0.07, 0.1],
    "num_layers": [1,2],
    "pool": None,
    }

BiGRUPoolEncoder_grid_stage1 = {
    "input_dim": [1024],
    "hidden_dim": [256],
    "output_dim": [256],
    "dropout": [0.0, 0.1],
    "lr": [5e-4, 2e-4],
    "weight_decay": [1e-4],
    "batch_size": [32],
    "temperature": [0.07, 0.1],
    "num_layers": [2],
    "pool": ["max", "mean"],
    }

NUM_EPOCHS = 20
results_csv_BiGRUPoolEncoder = "results/task3_cislr/bigru_pool_gridsearch_results.csv"
configs_BiGRUPoolEncoder = expand_grid(BiGRUPoolEncoder_grid_stage1)
results_csv_BiGRUAttentionEncoder = "results/task3_cislr/bigru_attention_gridsearch_results.csv"
configs_BiGRUAttentionEncoder = expand_grid(BiGRUAttentionEncoder_grid_stage1)
print(f"Total configs: {len(configs_BiGRUAttentionEncoder)}")
print("Sample config:", configs_BiGRUAttentionEncoder[0])

all_results = []

for i, cfg in enumerate(configs_BiGRUAttentionEncoder, start=1):
    print("\n" + "=" * 80)
    print(f"[{i}/{len(configs_BiGRUAttentionEncoder)}] Running BiGRUAttentionEncoder with config:")
    print(cfg)

    exp_name = (
        f"bigru_{cfg['pool']}_"
        f"h{cfg['hidden_dim']}_o{cfg['output_dim']}_"
        f"drop{str(cfg['dropout']).replace('.', 'p')}_"
        f"lr{cfg['lr']}_"
        f"wd{cfg['weight_decay']}_"
        f"temp{str(cfg['temperature']).replace('.', 'p')}_"
        f"layers{cfg['num_layers']}"
    )

    try:
        model = BiGRUPoolEncoder(
            input_dim=cfg["input_dim"],
            hidden_dim=cfg["hidden_dim"],
            out_dim=cfg["output_dim"],
            num_layers=cfg["num_layers"],
            pool=cfg["pool"],
            dropout=cfg["dropout"],
        )

        results = run_training(
            train_csv=ROOT / "dataset/task3_cislr/splits/train_repeated.csv",
            val_csv=ROOT / "dataset/task3_cislr/splits/val_repeated_no_proto_overlap.csv",
            prototype_csv=ROOT / "dataset/task3_cislr/splits/prototype.csv",
            test_csv=ROOT / "dataset/task3_cislr/splits/test.csv",
            feature_pkl=ROOT / "dataset/task3_cislr/features/I3D_features.pkl",
            out_dir=ROOT / "results/task3_cislr/BiGRUAttentionEncoder_gridsearch" / exp_name,
            batch_size=cfg["batch_size"],
            num_workers=0,
            pooling=cfg["pool"] if cfg["pool"] in ["max", "mean"] else None,
            lr=cfg["lr"],
            weight_decay=cfg["weight_decay"],
            temperature=cfg["temperature"],
            epochs=NUM_EPOCHS,
            early_stopping_patience=10,
            model=model,
        )

        row = {
            "exp_name": exp_name,
            **cfg,
            "best_val_top1": results["best_val_top1"],
            "val_top1": results["best_val_metrics"]["top1"],
            "val_top5": results["best_val_metrics"]["top5"],
            "val_top10": results["best_val_metrics"]["top10"],
            "test_top1": results["test_metrics"]["top1"],
            "test_top5": results["test_metrics"]["top5"],
            "test_top10": results["test_metrics"]["top10"],
        }
        all_results.append(row)

    except Exception as e:
        print(f"Failed config: {cfg}")
        print("Error:", e)
        all_results.append({
            "exp_name": exp_name,
            **cfg,
            "error": str(e),
        })

    df = pd.DataFrame(all_results)
    df.to_csv(ROOT / results_csv_BiGRUAttentionEncoder, 
              mode = "a", 
              header=not os.path.exists(ROOT / results_csv_BiGRUAttentionEncoder),
              index=False)

display(df.sort_values("test_top1", ascending=False))

Total configs: 16
Sample config: {'input_dim': 1024, 'hidden_dim': 256, 'output_dim': 256, 'dropout': 0.0, 'lr': 0.0005, 'weight_decay': 0.0001, 'batch_size': 32, 'temperature': 0.07, 'num_layers': 2, 'pool': 'max'}

[1/16] Running BiGRUPoolEncoder with config:
{'input_dim': 1024, 'hidden_dim': 256, 'output_dim': 256, 'dropout': 0.0, 'lr': 0.0005, 'weight_decay': 0.0001, 'batch_size': 32, 'temperature': 0.07, 'num_layers': 2, 'pool': 'max'}
Epoch 01 | loss=1.9499 | val_top1=7.61 | val_top5=13.04 | val_top10=15.22
Epoch 02 | loss=1.0470 | val_top1=9.78 | val_top5=11.96 | val_top10=14.13
Epoch 03 | loss=0.4665 | val_top1=11.96 | val_top5=15.22 | val_top10=16.30
Epoch 04 | loss=0.1758 | val_top1=9.78 | val_top5=14.13 | val_top10=19.57
Epoch 05 | loss=0.0449 | val_top1=10.87 | val_top5=19.57 | val_top10=21.74
Epoch 06 | loss=0.0096 | val_top1=11.96 | val_top5=18.48 | val_top10=20.65
Epoch 07 | loss=0.0024 | val_top1=14.13 | val_top5=18.48 | val_top10=20.65
Epoch 08 | loss=0.0019 | val_top1

,exp_name,input_dim,hidden_dim,output_dim,dropout,lr,weight_decay,batch_size,temperature,num_layers,pool,best_val_top1,val_top1,val_top5,val_top10,test_top1,test_top5,test_top10
6,bigru_max_h256_o256_drop0p0_lr0.0002_wd0.0001_...,1024,256,256,0.0,0.0002,0.0001,32,0.10,2,max,18.478261,18.478261,21.739130,22.826087,21.663020,26.477024,28.971554
14,bigru_max_h256_o256_drop0p1_lr0.0002_wd0.0001_...,1024,256,256,0.1,0.0002,0.0001,32,0.10,2,max,15.217391,15.217391,17.391304,20.652174,20.787746,25.601751,27.877462
2,bigru_max_h256_o256_drop0p0_lr0.0005_wd0.0001_...,1024,256,256,0.0,0.0005,0.0001,32,0.10,2,max,16.304348,16.304348,22.826087,23.913043,20.612691,25.295405,27.964989
12,bigru_max_h256_o256_drop0p1_lr0.0002_wd0.0001_...,1024,256,256,0.1,0.0002,0.0001,32,0.07,2,max,16.304348,16.304348,18.478261,20.652174,20.481400,25.645514,28.358862
7,bigru_mean_h256_o256_drop0p0_lr0.0002_wd0.0001...,1024,256,256,0.0,0.0002,0.0001,32,0.10,2,mean,14.130435,14.130435,18.478261,19.565217,20.350109,25.995624,28.577681
15,bigru_mean_h256_o256_drop0p1_lr0.0002_wd0.0001...,1024,256,256,0.1,0.0002,0.0001,32,0.10,2,mean,16.304348,16.304348,18.478261,21.739130,20.087527,25.032823,27.746171
4,bigru_max_h256_o256_drop0p0_lr0.0002_wd0.0001_...,1024,256,256,0.0,0.0002,0.0001,32,0.07,2,max,16.304348,16.304348,19.565217,23.913043,19.693654,25.864333,28.796499
10,bigru_max_h256_o256_drop0p1_lr0.0005_wd0.0001_...,1024,256,256,0.1,0.0005,0.0001,32,0.10,2,max,11.956522,11.956522,18.478261,20.652174,19.474836,24.463895,26.870897
5,bigru_mean_h256_o256_drop0p0_lr0.0002_wd0.0001...,1024,256,256,0.0,0.0002,0.0001,32,0.07,2,mean,15.217391,15.217391,19.565217,22.826087,19.387309,25.426696,28.665208
11,bigru_mean_h256_o256_drop0p1_lr0.0005_wd0.0001...,1024,256,256,0.1,0.0005,0.0001,32,0.10,2,mean,11.956522,11.956522,16.304348,17.391304,17.636761,22.013129,24.201313


In [23]:
from src.tasks.task3_cislr.models.bigru_encoder import BiGRUPoolEncoder, BiGRUAttentionEncoder
import os
import pandas as pd
BiGRUAttentionEncoder_grid_stage1 = {
    "input_dim": [1024],
    "hidden_dim": [256],
    "output_dim": [256],
    "dropout": [0.0, 0.1],
    "lr": [5e-4, 2e-4],
    "weight_decay": [1e-4],
    "batch_size": [32],
    "temperature": [0.07, 0.1],
    "num_layers": [1,2],
    #"pool": None,
    }

NUM_EPOCHS = 20
patience = 10
results_csv_BiGRUAttentionEncoder = "results/task3_cislr/bigru_attention_gridsearch_results.csv"
configs_BiGRUAttentionEncoder = expand_grid(BiGRUAttentionEncoder_grid_stage1)

print(f"Total configs: {len(configs_BiGRUAttentionEncoder)}")
print("Sample config:", configs_BiGRUAttentionEncoder[0])

all_results = []

for i, cfg in enumerate(configs_BiGRUAttentionEncoder, start=1):
    print("\n" + "=" * 80)
    print(f"[{i}/{len(configs_BiGRUAttentionEncoder)}] Running BiGRUAttentionEncoder with config:")
    print(cfg)

    exp_name = (
        f"bigru_attention_"
        f"h{cfg['hidden_dim']}_o{cfg['output_dim']}_"
        f"drop{str(cfg['dropout']).replace('.', 'p')}_"
        f"lr{cfg['lr']}_"
        f"wd{cfg['weight_decay']}_"
        f"temp{str(cfg['temperature']).replace('.', 'p')}_"
        f"layers{cfg['num_layers']}"
    )

    try:
        model = BiGRUAttentionEncoder(
            input_dim=cfg["input_dim"],
            hidden_dim=cfg["hidden_dim"],
            out_dim=cfg["output_dim"],
            num_layers=cfg["num_layers"],
            dropout=cfg["dropout"],
        )

        results = run_training(
            train_csv=ROOT / "dataset/task3_cislr/splits/train_repeated.csv",
            val_csv=ROOT / "dataset/task3_cislr/splits/val_repeated_no_proto_overlap.csv",
            prototype_csv=ROOT / "dataset/task3_cislr/splits/prototype.csv",
            test_csv=ROOT / "dataset/task3_cislr/splits/test.csv",
            feature_pkl=ROOT / "dataset/task3_cislr/features/I3D_features.pkl",
            out_dir=ROOT / "results/task3_cislr/BiGRUAttentionEncoder_gridsearch" / exp_name,
            batch_size=cfg["batch_size"],
            num_workers=0,
            pooling=None,
            lr=cfg["lr"],
            weight_decay=cfg["weight_decay"],
            temperature=cfg["temperature"],
            epochs=NUM_EPOCHS,
            early_stopping_patience=patience,
            model=model,
        )

        row = {
            "exp_name": exp_name,
            **cfg,
            "best_val_top1": results["best_val_top1"],
            "val_top1": results["best_val_metrics"]["top1"],
            "val_top5": results["best_val_metrics"]["top5"],
            "val_top10": results["best_val_metrics"]["top10"],
            "test_top1": results["test_metrics"]["top1"],
            "test_top5": results["test_metrics"]["top5"],
            "test_top10": results["test_metrics"]["top10"],
        }

    except Exception as e:
        print(f"Failed config: {cfg}")
        print("Error:", e)
        row = {
            "exp_name": exp_name,
            **cfg,
            "error": str(e),
        }

    all_results.append(row)

    pd.DataFrame([row]).to_csv(
        ROOT / results_csv_BiGRUAttentionEncoder,
        mode="a",
        header=not os.path.exists(ROOT / results_csv_BiGRUAttentionEncoder),
        index=False,
    )

df = pd.DataFrame(all_results)
display(df.sort_values("test_top1", ascending=False))

Total configs: 16
Sample config: {'input_dim': 1024, 'hidden_dim': 256, 'output_dim': 256, 'dropout': 0.0, 'lr': 0.0005, 'weight_decay': 0.0001, 'batch_size': 32, 'temperature': 0.07, 'num_layers': 1}

[1/16] Running BiGRUAttentionEncoder with config:
{'input_dim': 1024, 'hidden_dim': 256, 'output_dim': 256, 'dropout': 0.0, 'lr': 0.0005, 'weight_decay': 0.0001, 'batch_size': 32, 'temperature': 0.07, 'num_layers': 1}
Epoch 01 | loss=1.7869 | val_top1=10.87 | val_top5=13.04 | val_top10=14.13
Epoch 02 | loss=0.7711 | val_top1=11.96 | val_top5=14.13 | val_top10=17.39
Epoch 03 | loss=0.2436 | val_top1=11.96 | val_top5=15.22 | val_top10=17.39
Epoch 04 | loss=0.0354 | val_top1=11.96 | val_top5=19.57 | val_top10=21.74
Epoch 05 | loss=0.0062 | val_top1=11.96 | val_top5=19.57 | val_top10=21.74
Epoch 06 | loss=0.0037 | val_top1=11.96 | val_top5=18.48 | val_top10=22.83
Epoch 07 | loss=0.0031 | val_top1=11.96 | val_top5=18.48 | val_top10=21.74
Epoch 08 | loss=0.0026 | val_top1=11.96 | val_top5=18.4

,exp_name,input_dim,hidden_dim,output_dim,dropout,lr,weight_decay,batch_size,temperature,num_layers,best_val_top1,val_top1,val_top5,val_top10,test_top1,test_top5,test_top10
6,bigru_attention_h256_o256_drop0p0_lr0.0002_wd0...,1024,256,256,0.0,0.0002,0.0001,32,0.10,1,16.304348,16.304348,20.652174,21.739130,21.356674,27.045952,29.365427
14,bigru_attention_h256_o256_drop0p1_lr0.0002_wd0...,1024,256,256,0.1,0.0002,0.0001,32,0.10,1,17.391304,17.391304,19.565217,23.913043,20.393873,25.908096,28.315098
4,bigru_attention_h256_o256_drop0p0_lr0.0002_wd0...,1024,256,256,0.0,0.0002,0.0001,32,0.07,1,16.304348,16.304348,20.652174,21.739130,19.737418,26.477024,29.671772
3,bigru_attention_h256_o256_drop0p0_lr0.0005_wd0...,1024,256,256,0.0,0.0005,0.0001,32,0.10,2,13.043478,13.043478,17.391304,19.565217,19.562363,23.676149,25.820569
12,bigru_attention_h256_o256_drop0p1_lr0.0002_wd0...,1024,256,256,0.1,0.0002,0.0001,32,0.07,1,16.304348,16.304348,21.739130,21.739130,19.387309,26.039387,28.752735
0,bigru_attention_h256_o256_drop0p0_lr0.0005_wd0...,1024,256,256,0.0,0.0005,0.0001,32,0.07,1,14.130435,14.130435,20.652174,22.826087,19.037199,25.470460,28.183807
2,bigru_attention_h256_o256_drop0p0_lr0.0005_wd0...,1024,256,256,0.0,0.0005,0.0001,32,0.10,1,15.217391,15.217391,17.391304,22.826087,19.037199,25.032823,28.533917
15,bigru_attention_h256_o256_drop0p1_lr0.0002_wd0...,1024,256,256,0.1,0.0002,0.0001,32,0.10,2,16.304348,16.304348,21.739130,22.826087,18.862144,24.682713,27.483589
8,bigru_attention_h256_o256_drop0p1_lr0.0005_wd0...,1024,256,256,0.1,0.0005,0.0001,32,0.07,1,14.130435,14.130435,18.478261,23.913043,18.468271,24.551422,27.045952
13,bigru_attention_h256_o256_drop0p1_lr0.0002_wd0...,1024,256,256,0.1,0.0002,0.0001,32,0.07,2,15.217391,15.217391,17.391304,21.739130,18.161926,24.070022,27.527352


In [5]:
from src.utils.set_seeds import set_seed
from src.utils.expand_grid import expand_grid
from src.tasks.task3_cislr.models.bigru_encoder import BiGRUPoolEncoder, BiGRUAttentionEncoder
from src.tasks.task3_cislr.training.train_retreival import run_training

import os
import pandas as pd

seeds = [1990,42,2024]
BiGRUPoolEncoder_grid_stage1 = {
    "input_dim": [1024],
    "hidden_dim": [128,256,512],
    "output_dim": [128,256,512],
    "dropout": [0.0],
    "lr": [2e-4],
    "weight_decay": [1e-4],
    "batch_size": [32],
    "temperature": [0.1],
    "num_layers": [1],
    "pool": ["max"],
    }
NUM_EPOCHS = 20
patience = 10

results_csv_BiGRUPoolEncoder_with_seeds = "results/task3_cislr/bigru_pool_gridsearch_results_with_seeds.csv"
configs_BiGRUPoolEncoder = expand_grid(BiGRUPoolEncoder_grid_stage1)

print(f"Total configs: {len(configs_BiGRUPoolEncoder)}")
print("Sample config:", configs_BiGRUPoolEncoder[0])

all_results = []

for seed in seeds:
    print("\n" + "=" * 80)
    print(f"Running seed: {seed}")
    set_seed(seed)

    for i, cfg in enumerate(configs_BiGRUPoolEncoder, start=1):
        print("\n" + "=" * 80)
        print(f"[{i}/{len(configs_BiGRUPoolEncoder)}] Running BiGRUPoolEncoder with config:")
        print(cfg)

        exp_name = (
            f"bigru_pool_"
            f"h{cfg['hidden_dim']}_o{cfg['output_dim']}_"
            f"drop{str(cfg['dropout']).replace('.', 'p')}_"
            f"lr{cfg['lr']}_"
            f"wd{cfg['weight_decay']}_"
            f"temp{str(cfg['temperature']).replace('.', 'p')}_"
            f"layers{cfg['num_layers']}_"
            f"seed{seed}"
        )

        try:
            model = BiGRUPoolEncoder(
                input_dim=cfg["input_dim"],
                hidden_dim=cfg["hidden_dim"],
                out_dim=cfg["output_dim"],
                num_layers=cfg["num_layers"],
                pool=cfg["pool"],
                dropout=cfg["dropout"],
            )

            results = run_training(
                train_csv=ROOT / "dataset/task3_cislr/splits/train_repeated.csv",
                val_csv=ROOT / "dataset/task3_cislr/splits/val_repeated_no_proto_overlap.csv",
                prototype_csv=ROOT / "dataset/task3_cislr/splits/prototype.csv",
                test_csv=ROOT / "dataset/task3_cislr/splits/test.csv",
                feature_pkl=ROOT / "dataset/task3_cislr/features/I3D_features.pkl",
                out_dir=ROOT / "results/task3_cislr/BiGRUPoolEncoder_gridsearch" / exp_name,
                batch_size=cfg["batch_size"],
                num_workers=0,
                pooling=cfg["pool"] if cfg["pool"] in ["max", "mean"] else None,
                lr=cfg["lr"],
                weight_decay=cfg["weight_decay"],
                temperature=cfg["temperature"],
                epochs=NUM_EPOCHS,
                early_stopping_patience=None,
                model=model,
            )

            row = {
                "exp_name": exp_name,
                **cfg,
                "best_val_top1": results["best_val_top1"],
                "val_top1": results["best_val_metrics"]["top1"],
                "val_top5": results["best_val_metrics"]["top5"],
                "val_top10": results["best_val_metrics"]["top10"],
                "test_top1": results["test_metrics"]["top1"],
                "test_top5": results["test_metrics"]["top5"],
                "test_top10": results["test_metrics"]["top10"],
                "seed": seed,
            }

        except Exception as e:
            print(f"Failed config: {cfg}")
            print("Error:", e)
            row = {
                "exp_name": exp_name,
                **cfg,
                "seed": seed,
                "error": str(e),
            }

        all_results.append(row)

        pd.DataFrame([row]).to_csv(
            ROOT / results_csv_BiGRUPoolEncoder_with_seeds,
            mode="a",
            header=not os.path.exists(ROOT / results_csv_BiGRUPoolEncoder_with_seeds),
            index=False,
        )

df = pd.DataFrame(all_results)
display(df.sort_values("test_top1", ascending=False))

Total configs: 9
Sample config: {'input_dim': 1024, 'hidden_dim': 128, 'output_dim': 128, 'dropout': 0.0, 'lr': 0.0002, 'weight_decay': 0.0001, 'batch_size': 32, 'temperature': 0.1, 'num_layers': 1, 'pool': 'max'}

Running seed: 1990

[1/9] Running BiGRUPoolEncoder with config:
{'input_dim': 1024, 'hidden_dim': 128, 'output_dim': 128, 'dropout': 0.0, 'lr': 0.0002, 'weight_decay': 0.0001, 'batch_size': 32, 'temperature': 0.1, 'num_layers': 1, 'pool': 'max'}
Epoch 01 | loss=1.8655 | val_top1=9.78 | val_top5=15.22 | val_top10=16.30
Epoch 02 | loss=0.7507 | val_top1=10.87 | val_top5=16.30 | val_top10=17.39
Epoch 03 | loss=0.2738 | val_top1=10.87 | val_top5=19.57 | val_top10=19.57
Epoch 04 | loss=0.1153 | val_top1=13.04 | val_top5=19.57 | val_top10=19.57
Epoch 05 | loss=0.0690 | val_top1=14.13 | val_top5=18.48 | val_top10=19.57
Epoch 06 | loss=0.0506 | val_top1=15.22 | val_top5=18.48 | val_top10=20.65
Epoch 07 | loss=0.0401 | val_top1=15.22 | val_top5=17.39 | val_top10=20.65
Epoch 08 | loss

,exp_name,input_dim,hidden_dim,output_dim,dropout,lr,weight_decay,batch_size,temperature,num_layers,pool,best_val_top1,val_top1,val_top5,val_top10,test_top1,test_top5,test_top10,seed
26,bigru_pool_h512_o512_drop0p0_lr0.0002_wd0.0001...,1024,512,512,0.0,0.0002,0.0001,32,0.1,1,max,19.565217,19.565217,20.652174,25.000000,23.019694,28.577681,31.422319,2024
15,bigru_pool_h512_o128_drop0p0_lr0.0002_wd0.0001...,1024,512,128,0.0,0.0002,0.0001,32,0.1,1,max,17.391304,17.391304,23.913043,25.000000,22.932166,27.789934,29.759300,42
16,bigru_pool_h512_o256_drop0p0_lr0.0002_wd0.0001...,1024,512,256,0.0,0.0002,0.0001,32,0.1,1,max,18.478261,18.478261,21.739130,23.913043,22.888403,28.315098,30.809628,42
13,bigru_pool_h256_o256_drop0p0_lr0.0002_wd0.0001...,1024,256,256,0.0,0.0002,0.0001,32,0.1,1,max,20.652174,20.652174,22.826087,22.826087,22.450766,27.702407,30.328228,42
22,bigru_pool_h256_o256_drop0p0_lr0.0002_wd0.0001...,1024,256,256,0.0,0.0002,0.0001,32,0.1,1,max,18.478261,18.478261,21.739130,22.826087,22.319475,27.527352,30.109409,2024
23,bigru_pool_h256_o512_drop0p0_lr0.0002_wd0.0001...,1024,256,512,0.0,0.0002,0.0001,32,0.1,1,max,18.478261,18.478261,22.826087,25.000000,22.275711,28.271335,31.072210,2024
5,bigru_pool_h256_o512_drop0p0_lr0.0002_wd0.0001...,1024,256,512,0.0,0.0002,0.0001,32,0.1,1,max,18.478261,18.478261,20.652174,22.826087,22.275711,27.658643,30.328228,1990
7,bigru_pool_h512_o256_drop0p0_lr0.0002_wd0.0001...,1024,512,256,0.0,0.0002,0.0001,32,0.1,1,max,18.478261,18.478261,21.739130,22.826087,22.231947,27.746171,30.371991,1990
14,bigru_pool_h256_o512_drop0p0_lr0.0002_wd0.0001...,1024,256,512,0.0,0.0002,0.0001,32,0.1,1,max,20.652174,20.652174,21.739130,21.739130,22.188184,28.052516,30.722101,42
12,bigru_pool_h256_o128_drop0p0_lr0.0002_wd0.0001...,1024,256,128,0.0,0.0002,0.0001,32,0.1,1,max,17.391304,17.391304,18.478261,21.739130,22.188184,27.133479,29.671772,42


In [6]:
import pandas as pd

csv_path = ROOT / "results/task3_cislr/bigru_pool_gridsearch_results_with_seeds.csv"
df = pd.read_csv(csv_path)

group_cols = [
    "input_dim",
    "hidden_dim",
    "output_dim",
    "dropout",
    "lr",
    "weight_decay",
    "batch_size",
    "temperature",
    "num_layers",
    "pool",
]

summary = (
    df.groupby(group_cols)
      .agg(
          runs=("seed", "count"),
          mean_test_top1=("test_top1", "mean"),
          std_test_top1=("test_top1", "std"),
          mean_test_top5=("test_top5", "mean"),
          std_test_top5=("test_top5", "std"),
          mean_test_top10=("test_top10", "mean"),
          std_test_top10=("test_top10", "std"),
          best_single_run=("test_top1", "max"),
      )
      .reset_index()
      .sort_values("mean_test_top1", ascending=False)
)

display(summary)

ParserError: Error tokenizing data. C error: Expected 13 fields in line 29, saw 19


In [ ]:
summary["test_top1_mean_std"] = summary.apply(
    lambda r: f"{r['mean_test_top1']:.2f} ± {0.0 if pd.isna(r['std_test_top1']) else r['std_test_top1']:.2f}",
    axis=1,
)
summary["test_top5_mean_std"] = summary.apply(
    lambda r: f"{r['mean_test_top5']:.2f} ± {0.0 if pd.isna(r['std_test_top5']) else r['std_test_top5']:.2f}",
    axis=1,
)
summary["test_top10_mean_std"] = summary.apply(
    lambda r: f"{r['mean_test_top10']:.2f} ± {0.0 if pd.isna(r['std_test_top10']) else r['std_test_top10']:.2f}",
    axis=1,
)

leaderboard = summary[
    [
        "hidden_dim",
        "output_dim",
        "dropout",
        "lr",
        "weight_decay",
        "temperature",
        "num_layers",
        "pool",
        "n_runs",
        "test_top1_mean_std",
        "test_top5_mean_std",
        "test_top10_mean_std",
        "best_single_test_top1",
    ]
].sort_values("mean_test_top1", ascending=False)

display(leaderboard)

In [7]:
csv_path = ROOT / "results/task3_cislr/bigru_pool_gridsearch_results_with_seeds.csv"

with open(csv_path, "r", encoding="utf-8") as f:
    for i, line in enumerate(f, start=1):
        if i <= 35:
            print(f"LINE {i}: {line.rstrip()}")

LINE 1: exp_name,input_dim,hidden_dim,output_dim,dropout,lr,weight_decay,batch_size,temperature,num_layers,pool,seed,error
LINE 2: bigru_pool_h128_o128_drop0p0_lr0.0002_wd0.0001_temp0p1_layers1_seed1990,1024,128,128,0.0,0.0002,0.0001,32,0.1,1,max,1990,name 'run_training' is not defined
LINE 3: bigru_pool_h128_o256_drop0p0_lr0.0002_wd0.0001_temp0p1_layers1_seed1990,1024,128,256,0.0,0.0002,0.0001,32,0.1,1,max,1990,name 'run_training' is not defined
LINE 4: bigru_pool_h128_o512_drop0p0_lr0.0002_wd0.0001_temp0p1_layers1_seed1990,1024,128,512,0.0,0.0002,0.0001,32,0.1,1,max,1990,name 'run_training' is not defined
LINE 5: bigru_pool_h256_o128_drop0p0_lr0.0002_wd0.0001_temp0p1_layers1_seed1990,1024,256,128,0.0,0.0002,0.0001,32,0.1,1,max,1990,name 'run_training' is not defined
LINE 6: bigru_pool_h256_o256_drop0p0_lr0.0002_wd0.0001_temp0p1_layers1_seed1990,1024,256,256,0.0,0.0002,0.0001,32,0.1,1,max,1990,name 'run_training' is not defined
LINE 7: bigru_pool_h256_o512_drop0p0_lr0.0002_wd0.0001_te

In [8]:
csv_path = ROOT / "results/task3_cislr/bigru_pool_gridsearch_results_with_seeds.csv"

with open(csv_path, "r", encoding="utf-8") as f:
    for i, line in enumerate(f, start=1):
        n_fields = len(line.rstrip("\n").split(","))
        print(f"Line {i}: {n_fields} fields")

Line 1: 13 fields
Line 2: 13 fields
Line 3: 13 fields
Line 4: 13 fields
Line 5: 13 fields
Line 6: 13 fields
Line 7: 13 fields
Line 8: 13 fields
Line 9: 13 fields
Line 10: 13 fields
Line 11: 13 fields
Line 12: 13 fields
Line 13: 13 fields
Line 14: 13 fields
Line 15: 13 fields
Line 16: 13 fields
Line 17: 13 fields
Line 18: 13 fields
Line 19: 13 fields
Line 20: 13 fields
Line 21: 13 fields
Line 22: 13 fields
Line 23: 13 fields
Line 24: 13 fields
Line 25: 13 fields
Line 26: 13 fields
Line 27: 13 fields
Line 28: 13 fields
Line 29: 19 fields
Line 30: 19 fields
Line 31: 19 fields
Line 32: 19 fields
Line 33: 19 fields
Line 34: 19 fields
Line 35: 19 fields
Line 36: 19 fields
Line 37: 19 fields
Line 38: 19 fields
Line 39: 19 fields
Line 40: 19 fields
Line 41: 19 fields
Line 42: 19 fields
Line 43: 19 fields
Line 44: 19 fields
Line 45: 19 fields
Line 46: 19 fields
Line 47: 19 fields
Line 48: 19 fields
Line 49: 19 fields
Line 50: 19 fields
Line 51: 19 fields
Line 52: 19 fields
Line 53: 19 fields
Li

In [9]:
from pathlib import Path

csv_path = ROOT / "results/task3_cislr/bigru_pool_gridsearch_results_with_seeds.csv"
clean_path = ROOT / "results/task3_cislr/bigru_pool_gridsearch_results_with_seeds_clean.csv"

with open(csv_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

# Keep only the correct 19-column header + 19-column rows
new_header = (
    "exp_name,input_dim,hidden_dim,output_dim,dropout,lr,weight_decay,"
    "batch_size,temperature,num_layers,pool,best_val_top1,val_top1,val_top5,"
    "val_top10,test_top1,test_top5,test_top10,seed\n"
)

clean_lines = [new_header]

for line in lines[1:]:
    if len(line.rstrip("\n").split(",")) == 19:
        clean_lines.append(line)

with open(clean_path, "w", encoding="utf-8") as f:
    f.writelines(clean_lines)

print(f"Clean file saved to: {clean_path}")
print(f"Rows kept: {len(clean_lines) - 1}")

Clean file saved to: /Users/lokeshkumar/eMasters/EE964_Projects/results/task3_cislr/bigru_pool_gridsearch_results_with_seeds_clean.csv
Rows kept: 27


In [11]:
df = pd.read_csv(clean_path)
display(df.head())
print(df.shape)

,exp_name,input_dim,hidden_dim,output_dim,dropout,lr,weight_decay,batch_size,temperature,num_layers,pool,best_val_top1,val_top1,val_top5,val_top10,test_top1,test_top5,test_top10,seed
0,bigru_pool_h128_o128_drop0p0_lr0.0002_wd0.0001...,1024,128,128,0.0,0.0002,0.0001,32,0.1,1,max,15.217391,15.217391,18.478261,20.652174,18.687090,24.376368,27.658643,1990
1,bigru_pool_h128_o256_drop0p0_lr0.0002_wd0.0001...,1024,128,256,0.0,0.0002,0.0001,32,0.1,1,max,16.304348,16.304348,21.739130,22.826087,21.006565,26.958425,29.452954,1990
2,bigru_pool_h128_o512_drop0p0_lr0.0002_wd0.0001...,1024,128,512,0.0,0.0002,0.0001,32,0.1,1,max,18.478261,18.478261,21.739130,23.913043,21.050328,26.695842,29.890591,1990
3,bigru_pool_h256_o128_drop0p0_lr0.0002_wd0.0001...,1024,256,128,0.0,0.0002,0.0001,32,0.1,1,max,17.391304,17.391304,21.739130,22.826087,21.444201,27.089716,29.759300,1990
4,bigru_pool_h256_o256_drop0p0_lr0.0002_wd0.0001...,1024,256,256,0.0,0.0002,0.0001,32,0.1,1,max,17.391304,17.391304,22.826087,23.913043,22.100656,27.177243,30.021882,1990


(27, 19)


In [12]:
group_cols = [
    "input_dim",
    "hidden_dim",
    "output_dim",
    "dropout",
    "lr",
    "weight_decay",
    "batch_size",
    "temperature",
    "num_layers",
    "pool",
]

summary = (
    df.groupby(group_cols)
      .agg(
          runs=("seed", "count"),
          mean_test_top1=("test_top1", "mean"),
          std_test_top1=("test_top1", "std"),
          mean_test_top5=("test_top5", "mean"),
          std_test_top5=("test_top5", "std"),
          mean_test_top10=("test_top10", "mean"),
          std_test_top10=("test_top10", "std"),
          best_single_run=("test_top1", "max"),
      )
      .reset_index()
      .sort_values("mean_test_top1", ascending=False)
)

display(summary)

,input_dim,hidden_dim,output_dim,dropout,lr,weight_decay,batch_size,temperature,num_layers,pool,runs,mean_test_top1,std_test_top1,mean_test_top5,std_test_top5,mean_test_top10,std_test_top10,best_single_run
4,1024,256,256,0.0,0.0002,0.0001,32,0.1,1,max,3,22.290299,0.176869,27.469001,0.267400,30.153173,0.157792,22.450766
5,1024,256,512,0.0,0.0002,0.0001,32,0.1,1,max,3,22.246535,0.050534,27.994165,0.310486,30.707513,0.372206,22.275711
6,1024,512,128,0.0,0.0002,0.0001,32,0.1,1,max,3,22.246535,0.597392,27.556528,0.482063,29.934354,0.463151,22.932166
7,1024,512,256,0.0,0.0002,0.0001,32,0.1,1,max,3,22.056893,0.931457,27.629468,0.750816,30.342815,0.482063,22.888403
8,1024,512,512,0.0,0.0002,0.0001,32,0.1,1,max,3,21.633844,1.318975,27.775346,0.697936,30.605398,0.728811,23.019694
3,1024,256,128,0.0,0.0002,0.0001,32,0.1,1,max,3,21.269147,1.017917,26.827133,0.493192,29.336251,0.658397,22.188184
1,1024,128,256,0.0,0.0002,0.0001,32,0.1,1,max,3,20.773158,0.220272,26.812546,0.133700,29.175784,0.480072,21.006565
2,1024,128,512,0.0,0.0002,0.0001,32,0.1,1,max,3,20.510576,0.467951,26.374909,0.281361,29.277899,0.612691,21.050328
0,1024,128,128,0.0,0.0002,0.0001,32,0.1,1,max,3,19.824945,1.208860,25.251641,0.974663,28.067104,0.707475,21.094092


In [13]:
from src.tasks.task3_cislr.models.bigru_encoder import BiGRUAttentionEncoder
from src.utils.set_seeds import set_seed
import pandas as pd
import os

seeds = [1990, 42, 2024]

BiGRUAttentionEncoder_grid = {
    "input_dim": [1024],
    "hidden_dim": [256],
    "output_dim": [256],
    "dropout": [0.0],
    "lr": [2e-4],
    "weight_decay": [1e-4],
    "batch_size": [32],
    "temperature": [0.1],
    "num_layers": [1],
}

NUM_EPOCHS = 20

results_csv_BiGRUAttentionEncoder_with_seeds = (
    "results/task3_cislr/bigru_attention_gridsearch_results_with_seeds.csv"
)

configs_BiGRUAttentionEncoder = expand_grid(BiGRUAttentionEncoder_grid)

print(f"Total configs: {len(configs_BiGRUAttentionEncoder)}")
print("Sample config:", configs_BiGRUAttentionEncoder[0])

all_results = []

for seed in seeds:
    print("\n" + "=" * 80)
    print(f"Running seed: {seed}")
    set_seed(seed)

    for i, cfg in enumerate(configs_BiGRUAttentionEncoder, start=1):
        print("\n" + "=" * 80)
        print(f"[{i}/{len(configs_BiGRUAttentionEncoder)}] Running BiGRUAttentionEncoder with config:")
        print(cfg)

        exp_name = (
            f"bigru_attention_"
            f"h{cfg['hidden_dim']}_o{cfg['output_dim']}_"
            f"drop{str(cfg['dropout']).replace('.', 'p')}_"
            f"lr{cfg['lr']}_"
            f"wd{cfg['weight_decay']}_"
            f"temp{str(cfg['temperature']).replace('.', 'p')}_"
            f"layers{cfg['num_layers']}_"
            f"seed{seed}"
        )

        try:
            model = BiGRUAttentionEncoder(
                input_dim=cfg["input_dim"],
                hidden_dim=cfg["hidden_dim"],
                out_dim=cfg["output_dim"],
                num_layers=cfg["num_layers"],
                dropout=cfg["dropout"],
            )

            results = run_training(
                train_csv=ROOT / "dataset/task3_cislr/splits/train_repeated.csv",
                val_csv=ROOT / "dataset/task3_cislr/splits/val_repeated_no_proto_overlap.csv",
                prototype_csv=ROOT / "dataset/task3_cislr/splits/prototype.csv",
                test_csv=ROOT / "dataset/task3_cislr/splits/test.csv",
                feature_pkl=ROOT / "dataset/task3_cislr/features/I3D_features.pkl",
                out_dir=ROOT / "results/task3_cislr/BiGRUAttentionEncoder_gridsearch" / exp_name,
                batch_size=cfg["batch_size"],
                num_workers=0,
                pooling="attention",
                lr=cfg["lr"],
                weight_decay=cfg["weight_decay"],
                temperature=cfg["temperature"],
                epochs=NUM_EPOCHS,
                early_stopping_patience=None,
                model=model,
            )

            row = {
                "exp_name": exp_name,
                **cfg,
                "pool": "attention",
                "best_val_top1": results["best_val_top1"],
                "val_top1": results["best_val_metrics"]["top1"],
                "val_top5": results["best_val_metrics"]["top5"],
                "val_top10": results["best_val_metrics"]["top10"],
                "test_top1": results["test_metrics"]["top1"],
                "test_top5": results["test_metrics"]["top5"],
                "test_top10": results["test_metrics"]["top10"],
                "seed": seed,
            }

        except Exception as e:
            print(f"Failed config: {cfg}")
            print("Error:", e)
            row = {
                "exp_name": exp_name,
                **cfg,
                "pool": "attention",
                "seed": seed,
                "error": str(e),
            }

        all_results.append(row)

        pd.DataFrame([row]).to_csv(
            ROOT / results_csv_BiGRUAttentionEncoder_with_seeds,
            mode="a",
            header=not os.path.exists(ROOT / results_csv_BiGRUAttentionEncoder_with_seeds),
            index=False,
        )

df = pd.DataFrame(all_results)
display(df.sort_values("test_top1", ascending=False))

Total configs: 1
Sample config: {'input_dim': 1024, 'hidden_dim': 256, 'output_dim': 256, 'dropout': 0.0, 'lr': 0.0002, 'weight_decay': 0.0001, 'batch_size': 32, 'temperature': 0.1, 'num_layers': 1}

Running seed: 1990

[1/1] Running BiGRUAttentionEncoder with config:
{'input_dim': 1024, 'hidden_dim': 256, 'output_dim': 256, 'dropout': 0.0, 'lr': 0.0002, 'weight_decay': 0.0001, 'batch_size': 32, 'temperature': 0.1, 'num_layers': 1}
Epoch 01 | loss=1.7295 | val_top1=10.87 | val_top5=14.13 | val_top10=18.48
Epoch 02 | loss=0.6731 | val_top1=13.04 | val_top5=16.30 | val_top10=17.39
Epoch 03 | loss=0.2254 | val_top1=14.13 | val_top5=17.39 | val_top10=21.74
Epoch 04 | loss=0.0840 | val_top1=14.13 | val_top5=19.57 | val_top10=23.91
Epoch 05 | loss=0.0481 | val_top1=14.13 | val_top5=20.65 | val_top10=23.91
Epoch 06 | loss=0.0357 | val_top1=14.13 | val_top5=20.65 | val_top10=25.00
Epoch 07 | loss=0.0284 | val_top1=15.22 | val_top5=21.74 | val_top10=25.00
Epoch 08 | loss=0.0234 | val_top1=16.30

,exp_name,input_dim,hidden_dim,output_dim,dropout,lr,weight_decay,batch_size,temperature,num_layers,pool,best_val_top1,val_top1,val_top5,val_top10,test_top1,test_top5,test_top10,seed
2,bigru_attention_h256_o256_drop0p0_lr0.0002_wd0...,1024,256,256,0.0,0.0002,0.0001,32,0.1,1,attention,16.304348,16.304348,20.652174,22.826087,22.056893,27.658643,30.459519,2024
0,bigru_attention_h256_o256_drop0p0_lr0.0002_wd0...,1024,256,256,0.0,0.0002,0.0001,32,0.1,1,attention,18.478261,18.478261,22.826087,26.086957,21.619256,27.571116,30.021882,1990
1,bigru_attention_h256_o256_drop0p0_lr0.0002_wd0...,1024,256,256,0.0,0.0002,0.0001,32,0.1,1,attention,17.391304,17.391304,20.652174,23.913043,20.525164,26.258206,29.496718,42


In [3]:
from src.tasks.task3_cislr.models.transformer_encoder import TemporalTransformerEncoder
from src.utils.set_seeds import set_seed
from src.utils.expand_grid import expand_grid
from src.tasks.task3_cislr.training.train_retreival import run_training
from src.scripts.get_device import get_device
import pandas as pd
import os

seeds = [1990, 42, 2024]

TemporalTransformerEncoder_grid = {
    "input_dim": [1024],
    "model_dim": [256, 512],
    "output_dim": [256],
    "num_heads": [4, 8],
    "ff_multiplier": [2, 4],
    "dropout": [0.0, 0.1],
    "lr": [2e-4, 1e-5],
    "weight_decay": [1e-4],
    "batch_size": [32],
    "temperature": [0.1],
    "num_layers": [1, 2],
    "pool": ["mean"]
}

NUM_EPOCHS = 20

results_csv_TemporalTransformerEncoder_with_seeds = (
    "results/task3_cislr/temporal_transformer_encoder_gridsearch_results_with_seeds.csv"
)

configs_TemporalTransformerEncoder = expand_grid(TemporalTransformerEncoder_grid)

print(f"Total configs: {len(configs_TemporalTransformerEncoder)}")
print("Sample config:", configs_TemporalTransformerEncoder[0])

all_results = []

for seed in seeds:
    print("\n" + "=" * 80)
    print(f"Running seed: {seed}")
    set_seed(seed)

    for i, cfg in enumerate(configs_TemporalTransformerEncoder, start=1):
        print("\n" + "=" * 80)
        print(f"[{i}/{len(configs_TemporalTransformerEncoder)}] Running TemporalTransformerEncoder with config:")
        print(cfg)

        exp_name = (
            f"temporal_transformer_encoder_"
            f"dim{cfg['model_dim']}_o{cfg['output_dim']}_"
            f"heads{cfg['num_heads']}_"
            f"ff{cfg['ff_multiplier']}_"
            f"drop{str(cfg['dropout']).replace('.', 'p')}_"
            f"lr{cfg['lr']}_"
            f"wd{cfg['weight_decay']}_"
            f"temp{str(cfg['temperature']).replace('.', 'p')}_"
            f"layers{cfg['num_layers']}_"
            f"seed{seed}"
        )

        try:
            model = TemporalTransformerEncoder(
                input_dim=cfg["input_dim"],
                model_dim=cfg["model_dim"],
                out_dim=cfg["output_dim"],
                num_heads=cfg["num_heads"],
                num_layers=cfg["num_layers"],
                ff_mult=cfg["ff_multiplier"],
                dropout=cfg["dropout"],
                pool=cfg["pool"],
            )

            results = run_training(
                train_csv=ROOT / "dataset/task3_cislr/splits/train_repeated.csv",
                val_csv=ROOT / "dataset/task3_cislr/splits/val_repeated_no_proto_overlap.csv",
                prototype_csv=ROOT / "dataset/task3_cislr/splits/prototype.csv",
                test_csv=ROOT / "dataset/task3_cislr/splits/test.csv",
                feature_pkl=ROOT / "dataset/task3_cislr/features/I3D_features.pkl",
                out_dir=ROOT / "results/task3_cislr/TemporalTransformerEncoder_gridsearch" / exp_name,
                batch_size=cfg["batch_size"],
                num_workers=0,
                pooling="mean",
                lr=cfg["lr"],
                weight_decay=cfg["weight_decay"],
                temperature=cfg["temperature"],
                epochs=NUM_EPOCHS,
                early_stopping_patience=None,
                model=model,
            )

            row = {
                "exp_name": exp_name,
                **cfg,
                "pool": "mean",
                "best_val_top1": results["best_val_top1"],
                "val_top1": results["best_val_metrics"]["top1"],
                "val_top5": results["best_val_metrics"]["top5"],
                "val_top10": results["best_val_metrics"]["top10"],
                "test_top1": results["test_metrics"]["top1"],
                "test_top5": results["test_metrics"]["top5"],
                "test_top10": results["test_metrics"]["top10"],
                "seed": seed,
            }

        except Exception as e:
            print(f"Failed config: {cfg}")
            print("Error:", e)
            row = {
                "exp_name": exp_name,
                **cfg,
                "pool": "mean",
                "seed": seed,
                "error": str(e),
            }

        all_results.append(row)

        pd.DataFrame([row]).to_csv(
            ROOT / results_csv_TemporalTransformerEncoder_with_seeds,
            mode="a",
            header=not os.path.exists(ROOT / results_csv_TemporalTransformerEncoder_with_seeds),
            index=False,
        )

df = pd.DataFrame(all_results)
display(df.sort_values("test_top1", ascending=False))

Total configs: 64
Sample config: {'input_dim': 1024, 'model_dim': 256, 'output_dim': 256, 'num_heads': 4, 'ff_multiplier': 2, 'dropout': 0.0, 'lr': 0.0002, 'weight_decay': 0.0001, 'batch_size': 32, 'temperature': 0.1, 'num_layers': 1, 'pool': 'mean'}

Running seed: 1990

[1/64] Running TemporalTransformerEncoder with config:
{'input_dim': 1024, 'model_dim': 256, 'output_dim': 256, 'num_heads': 4, 'ff_multiplier': 2, 'dropout': 0.0, 'lr': 0.0002, 'weight_decay': 0.0001, 'batch_size': 32, 'temperature': 0.1, 'num_layers': 1, 'pool': 'mean'}


/opt/anaconda3/envs/LLM/lib/python3.13/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


Epoch 01 | loss=1.8647 | val_top1=10.87 | val_top5=15.22 | val_top10=15.22
Epoch 02 | loss=0.9954 | val_top1=9.78 | val_top5=15.22 | val_top10=16.30
Epoch 03 | loss=0.4617 | val_top1=9.78 | val_top5=15.22 | val_top10=17.39
Epoch 04 | loss=0.1962 | val_top1=11.96 | val_top5=16.30 | val_top10=18.48
Epoch 05 | loss=0.0755 | val_top1=13.04 | val_top5=18.48 | val_top10=19.57
Epoch 06 | loss=0.0421 | val_top1=13.04 | val_top5=18.48 | val_top10=19.57
Epoch 07 | loss=0.0306 | val_top1=13.04 | val_top5=18.48 | val_top10=20.65
Epoch 08 | loss=0.0248 | val_top1=14.13 | val_top5=19.57 | val_top10=20.65
Epoch 09 | loss=0.0209 | val_top1=14.13 | val_top5=20.65 | val_top10=20.65
Epoch 10 | loss=0.0180 | val_top1=14.13 | val_top5=20.65 | val_top10=20.65
Epoch 11 | loss=0.0157 | val_top1=14.13 | val_top5=20.65 | val_top10=21.74
Epoch 12 | loss=0.0139 | val_top1=14.13 | val_top5=20.65 | val_top10=21.74
Epoch 13 | loss=0.0124 | val_top1=15.22 | val_top5=20.65 | val_top10=21.74
Epoch 14 | loss=0.0111 | va

/opt/anaconda3/envs/LLM/lib/python3.13/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


Epoch 01 | loss=1.9592 | val_top1=8.70 | val_top5=11.96 | val_top10=13.04
Epoch 02 | loss=1.1678 | val_top1=8.70 | val_top5=11.96 | val_top10=15.22
Epoch 03 | loss=0.6080 | val_top1=8.70 | val_top5=13.04 | val_top10=16.30
Epoch 04 | loss=0.2416 | val_top1=8.70 | val_top5=15.22 | val_top10=17.39
Epoch 05 | loss=0.0897 | val_top1=10.87 | val_top5=20.65 | val_top10=22.83
Epoch 06 | loss=0.0393 | val_top1=10.87 | val_top5=18.48 | val_top10=22.83
Epoch 07 | loss=0.0253 | val_top1=10.87 | val_top5=17.39 | val_top10=23.91
Epoch 08 | loss=0.0204 | val_top1=10.87 | val_top5=17.39 | val_top10=25.00
Epoch 09 | loss=0.0173 | val_top1=10.87 | val_top5=18.48 | val_top10=25.00
Epoch 10 | loss=0.0151 | val_top1=10.87 | val_top5=18.48 | val_top10=25.00
Epoch 11 | loss=0.0133 | val_top1=11.96 | val_top5=18.48 | val_top10=25.00
Epoch 12 | loss=0.0118 | val_top1=13.04 | val_top5=18.48 | val_top10=25.00
Epoch 13 | loss=0.0106 | val_top1=13.04 | val_top5=18.48 | val_top10=23.91
Epoch 14 | loss=0.0096 | val_

...

/opt/anaconda3/envs/LLM/lib/python3.13/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


Epoch 01 | loss=2.2150 | val_top1=10.87 | val_top5=11.96 | val_top10=17.39
Epoch 02 | loss=1.8099 | val_top1=10.87 | val_top5=13.04 | val_top10=16.30
Epoch 03 | loss=1.5646 | val_top1=9.78 | val_top5=14.13 | val_top10=18.48
Epoch 04 | loss=1.3725 | val_top1=9.78 | val_top5=14.13 | val_top10=18.48
Epoch 05 | loss=1.2228 | val_top1=9.78 | val_top5=14.13 | val_top10=17.39
Epoch 06 | loss=1.1147 | val_top1=9.78 | val_top5=14.13 | val_top10=17.39
Epoch 07 | loss=0.9940 | val_top1=9.78 | val_top5=14.13 | val_top10=17.39
Epoch 08 | loss=0.8915 | val_top1=9.78 | val_top5=15.22 | val_top10=16.30
Epoch 09 | loss=0.7819 | val_top1=10.87 | val_top5=15.22 | val_top10=17.39
Epoch 10 | loss=0.7082 | val_top1=10.87 | val_top5=15.22 | val_top10=15.22
Epoch 11 | loss=0.6249 | val_top1=10.87 | val_top5=15.22 | val_top10=16.30
Epoch 12 | loss=0.5687 | val_top1=10.87 | val_top5=15.22 | val_top10=15.22
Epoch 13 | loss=0.5143 | val_top1=10.87 | val_top5=15.22 | val_top10=16.30
Epoch 14 | loss=0.4609 | val_to

/opt/anaconda3/envs/LLM/lib/python3.13/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


Epoch 01 | loss=2.0388 | val_top1=11.96 | val_top5=15.22 | val_top10=15.22
Epoch 02 | loss=1.6558 | val_top1=13.04 | val_top5=14.13 | val_top10=15.22
Epoch 03 | loss=1.3945 | val_top1=13.04 | val_top5=15.22 | val_top10=15.22
Epoch 04 | loss=1.1694 | val_top1=11.96 | val_top5=15.22 | val_top10=16.30
Epoch 05 | loss=1.0093 | val_top1=11.96 | val_top5=15.22 | val_top10=16.30
Epoch 06 | loss=0.8603 | val_top1=10.87 | val_top5=14.13 | val_top10=17.39
Epoch 07 | loss=0.7709 | val_top1=10.87 | val_top5=14.13 | val_top10=17.39
Epoch 08 | loss=0.6773 | val_top1=10.87 | val_top5=15.22 | val_top10=18.48
Epoch 09 | loss=0.5753 | val_top1=10.87 | val_top5=15.22 | val_top10=18.48
Epoch 10 | loss=0.5158 | val_top1=10.87 | val_top5=16.30 | val_top10=18.48
Epoch 11 | loss=0.4560 | val_top1=10.87 | val_top5=16.30 | val_top10=18.48
Epoch 12 | loss=0.3907 | val_top1=10.87 | val_top5=17.39 | val_top10=18.48
Epoch 13 | loss=0.3592 | val_top1=9.78 | val_top5=18.48 | val_top10=18.48
Epoch 14 | loss=0.3073 | v

,exp_name,input_dim,model_dim,output_dim,num_heads,ff_multiplier,dropout,lr,weight_decay,batch_size,...,num_layers,pool,best_val_top1,val_top1,val_top5,val_top10,test_top1,test_top5,test_top10,seed
48,temporal_transformer_encoder_dim512_o256_heads...,1024,512,256,8,2,0.0,0.0002,0.0001,32,...,1,mean,16.304348,16.304348,19.565217,20.652174,19.824945,24.638950,26.695842,1990
24,temporal_transformer_encoder_dim256_o256_heads...,1024,256,256,8,4,0.0,0.0002,0.0001,32,...,1,mean,14.130435,14.130435,19.565217,22.826087,19.649891,24.551422,27.308534,1990
64,temporal_transformer_encoder_dim256_o256_heads...,1024,256,256,4,2,0.0,0.0002,0.0001,32,...,1,mean,14.130435,14.130435,19.565217,20.652174,19.606127,24.726477,27.089716,42
144,temporal_transformer_encoder_dim256_o256_heads...,1024,256,256,8,2,0.0,0.0002,0.0001,32,...,1,mean,18.478261,18.478261,20.652174,22.826087,19.518600,24.245077,26.433260,2024
120,temporal_transformer_encoder_dim512_o256_heads...,1024,512,256,8,4,0.0,0.0002,0.0001,32,...,1,mean,14.130435,14.130435,20.652174,22.826087,19.387309,24.245077,26.433260,42
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
177,temporal_transformer_encoder_dim512_o256_heads...,1024,512,256,8,2,0.0,0.0002,0.0001,32,...,2,mean,10.869565,10.869565,11.956522,13.043478,13.654267,17.417943,19.474836,2024
57,temporal_transformer_encoder_dim512_o256_heads...,1024,512,256,8,4,0.0,0.0002,0.0001,32,...,2,mean,11.956522,11.956522,13.043478,15.217391,13.566740,16.892779,19.518600,1990
105,temporal_transformer_encoder_dim512_o256_heads...,1024,512,256,4,4,0.0,0.0002,0.0001,32,...,2,mean,9.782609,9.782609,10.869565,10.869565,13.435449,16.105033,17.986871,42
125,temporal_transformer_encoder_dim512_o256_heads...,1024,512,256,8,4,0.1,0.0002,0.0001,32,...,2,mean,13.043478,13.043478,13.043478,14.130435,13.172867,16.805252,18.905908,42


In [5]:
import pandas as pd

csv_path = ROOT / "results/task3_cislr/temporal_transformer_encoder_gridsearch_results_with_seeds.csv"

df = pd.read_csv(csv_path)

# remove failed runs
df = df[df["error"].isna()].copy()

print("Remaining successful runs:", len(df))
df.head()

ParserError: Error tokenizing data. C error: Expected 15 fields in line 5, saw 21


In [6]:
csv_path = ROOT / "results/task3_cislr/temporal_transformer_encoder_gridsearch_results_with_seeds.csv"

with open(csv_path, "r", encoding="utf-8") as f:
    for i, line in enumerate(f, start=1):
        n_fields = line.count(",") + 1
        if n_fields != 15:
            print(f"Line {i}: fields={n_fields}")
            print(line[:500])
            print("-" * 100)

Line 5: fields=21
temporal_transformer_encoder_dim256_o256_heads4_ff2_drop0p0_lr0.0002_wd0.0001_temp0p1_layers1_seed1990,1024,256,256,4,2,0.0,0.0002,0.0001,32,0.1,1,mean,15.217391304347826,15.217391304347826,20.652173913043477,21.73913043478261,19.12472647702407,24.50765864332604,26.958424507658645,1990

----------------------------------------------------------------------------------------------------
Line 6: fields=21
temporal_transformer_encoder_dim256_o256_heads4_ff2_drop0p0_lr0.0002_wd0.0001_temp0p1_layers2_seed1990,1024,256,256,4,2,0.0,0.0002,0.0001,32,0.1,2,mean,15.217391304347826,15.217391304347826,18.47826086956522,21.73913043478261,18.643326039387308,23.150984682713347,25.601750547045953,1990

----------------------------------------------------------------------------------------------------
Line 7: fields=21
temporal_transformer_encoder_dim256_o256_heads4_ff2_drop0p0_lr1e-05_wd0.0001_temp0p1_layers1_seed1990,1024,256,256,4,2,0.0,1e-05,0.0001,32,0.1,1,mean,10.86956521739130

In [7]:
import pandas as pd

csv_path = ROOT / "results/task3_cislr/temporal_transformer_encoder_gridsearch_results_with_seeds.csv"

cols = [
    "exp_name",
    "input_dim",
    "model_dim",
    "output_dim",
    "num_heads",
    "ff_multiplier",
    "dropout",
    "lr",
    "weight_decay",
    "batch_size",
    "temperature",
    "num_layers",
    "pool",
    "best_val_top1",
    "val_top1",
    "val_top5",
    "val_top10",
    "test_top1",
    "test_top5",
    "test_top10",
    "seed",
]

df = pd.read_csv(csv_path, names=cols, header=0)

print("Loaded rows:", len(df))
print(df.head())
print(df.columns.tolist())

ParserError: Error tokenizing data. C error: Expected 15 fields in line 5, saw 21


In [12]:
import csv
from pathlib import Path
import pandas as pd
import math

raw_csv = ROOT / "results/task3_cislr/temporal_transformer_encoder_gridsearch_results_with_seeds.csv"
clean_csv = ROOT / "results/task3_cislr/temporal_transformer_encoder_gridsearch_results_clean.csv"

expected_cols = [
    "exp_name",
    "input_dim",
    "model_dim",
    "output_dim",
    "num_heads",
    "ff_multiplier",
    "dropout",
    "lr",
    "weight_decay",
    "batch_size",
    "temperature",
    "num_layers",
    "pool",
    "best_val_top1",
    "val_top1",
    "val_top5",
    "val_top10",
    "test_top1",
    "test_top5",
    "test_top10",
    "seed",
    "error",
]

numeric_cols = [
    "input_dim",
    "model_dim",
    "output_dim",
    "num_heads",
    "ff_multiplier",
    "dropout",
    "lr",
    "weight_decay",
    "batch_size",
    "temperature",
    "num_layers",
    "best_val_top1",
    "val_top1",
    "val_top5",
    "val_top10",
    "test_top1",
    "test_top5",
    "test_top10",
    "seed",
]

def to_num(x):
    if x is None:
        return None
    x = str(x).strip()
    if x == "" or x.lower() == "nan":
        return None
    try:
        val = float(x)
        return val
    except Exception:
        return None

def normalize_error(x):
    if x is None:
        return None
    x = str(x).strip()
    if x == "" or x.lower() == "nan":
        return None
    return x

rows = []
bad_rows = []

with open(raw_csv, "r", encoding="utf-8", newline="") as f:
    reader = csv.reader(f)
    header = next(reader, None)

    for line_idx, row in enumerate(reader, start=2):
        # Skip empty lines
        if not row or all(str(x).strip() == "" for x in row):
            continue

        # Case A: already correct width
        if len(row) == len(expected_cols):
            rec = dict(zip(expected_cols, row))
        # Case B: one extra leading unnamed index column
        elif len(row) == len(expected_cols) + 1:
            rec = dict(zip(expected_cols, row[1:]))
        # Case C: malformed rows from earlier error-only logging
        elif len(row) == 15:
            # exp_name ... pool, seed, error
            rec = {
                "exp_name": row[0],
                "input_dim": row[1],
                "model_dim": row[2],
                "output_dim": row[3],
                "num_heads": row[4],
                "ff_multiplier": row[5],
                "dropout": row[6],
                "lr": row[7],
                "weight_decay": row[8],
                "batch_size": row[9],
                "temperature": row[10],
                "num_layers": row[11],
                "pool": row[12],
                "best_val_top1": None,
                "val_top1": None,
                "val_top5": None,
                "val_top10": None,
                "test_top1": None,
                "test_top5": None,
                "test_top10": None,
                "seed": row[13],
                "error": row[14],
            }
        else:
            bad_rows.append({"line_idx": line_idx, "raw": row, "len": len(row)})
            continue

        # Clean types
        for col in numeric_cols:
            rec[col] = to_num(rec.get(col))

        rec["pool"] = None if rec.get("pool") in ("", None) else str(rec["pool"]).strip()
        rec["exp_name"] = None if rec.get("exp_name") in ("", None) else str(rec["exp_name"]).strip()
        rec["error"] = normalize_error(rec.get("error"))

        # Cast integer-like numeric fields
        for col in ["input_dim", "model_dim", "output_dim", "num_heads", "ff_multiplier", "batch_size", "num_layers", "seed"]:
            if rec[col] is not None and not math.isnan(rec[col]):
                rec[col] = int(rec[col])

        rows.append(rec)

df_clean = pd.DataFrame(rows, columns=expected_cols)

# Drop exact duplicates
df_clean = df_clean.drop_duplicates()

# Save cleaned file
df_clean.to_csv(clean_csv, index=False)

print("Clean rows:", len(df_clean))
print("Bad/unparsed rows:", len(bad_rows))
if bad_rows:
    print("Example bad row:", bad_rows[0])

display(df_clean.head())

Clean rows: 2
Bad/unparsed rows: 192
Example bad row: {'line_idx': 5, 'raw': ['temporal_transformer_encoder_dim256_o256_heads4_ff2_drop0p0_lr0.0002_wd0.0001_temp0p1_layers1_seed1990', '1024', '256', '256', '4', '2', '0.0', '0.0002', '0.0001', '32', '0.1', '1', 'mean', '15.217391304347826', '15.217391304347826', '20.652173913043477', '21.73913043478261', '19.12472647702407', '24.50765864332604', '26.958424507658645', '1990'], 'len': 21}


,exp_name,input_dim,model_dim,output_dim,num_heads,ff_multiplier,dropout,lr,weight_decay,batch_size,...,pool,best_val_top1,val_top1,val_top5,val_top10,test_top1,test_top5,test_top10,seed,error
0,temporal_transformer_encoder_dim256_o256_heads...,1024,256,256,4,2,0.0,0.0002,0.0001,32,...,mean,None,None,None,None,None,None,None,1990,'TemporalTransformerEncoder' object has no att...
2,temporal_transformer_encoder_dim256_o256_heads...,1024,256,256,4,2,0.0,0.0002,0.0001,32,...,mean,None,None,None,None,None,None,None,1990,masked_pool() got an unexpected keyword argume...


In [13]:
summary_csv = ROOT / "results/task3_cislr/temporal_transformer_gridsearch_summary.csv"

df = pd.read_csv(clean_csv)

# Successful runs only
df_ok = df[df["error"].isna()].copy()

group_cols = [
    "input_dim",
    "model_dim",
    "output_dim",
    "num_heads",
    "ff_multiplier",
    "dropout",
    "lr",
    "weight_decay",
    "batch_size",
    "temperature",
    "num_layers",
    "pool",
]

summary = (
    df_ok.groupby(group_cols, dropna=False)
    .agg(
        runs=("seed", "count"),
        mean_val_top1=("val_top1", "mean"),
        std_val_top1=("val_top1", "std"),
        mean_test_top1=("test_top1", "mean"),
        std_test_top1=("test_top1", "std"),
        mean_test_top5=("test_top5", "mean"),
        std_test_top5=("test_top5", "std"),
        mean_test_top10=("test_top10", "mean"),
        std_test_top10=("test_top10", "std"),
        best_single_run_top1=("test_top1", "max"),
    )
    .reset_index()
    .sort_values(["mean_test_top1", "mean_test_top5", "mean_test_top10"], ascending=False)
)

summary.to_csv(summary_csv, index=False)

print("Successful runs:", len(df_ok))
print("Unique configs:", len(summary))
display(summary.head(10))

Successful runs: 0
Unique configs: 0


,input_dim,model_dim,output_dim,num_heads,ff_multiplier,dropout,lr,weight_decay,batch_size,temperature,...,runs,mean_val_top1,std_val_top1,mean_test_top1,std_test_top1,mean_test_top5,std_test_top5,mean_test_top10,std_test_top10,best_single_run_top1


In [11]:
topk = 10  # change if needed

summary_md = summary.copy()

for col in [
    "mean_test_top1", "std_test_top1",
    "mean_test_top5", "std_test_top5",
    "mean_test_top10", "std_test_top10",
    "best_single_run_top1",
]:
    summary_md[col] = summary_md[col].round(2)

summary_md["Top1 (mean±std)"] = summary_md.apply(
    lambda r: f"{r['mean_test_top1']:.2f} ± {0.0 if pd.isna(r['std_test_top1']) else r['std_test_top1']:.2f}",
    axis=1
)
summary_md["Top5 (mean±std)"] = summary_md.apply(
    lambda r: f"{r['mean_test_top5']:.2f} ± {0.0 if pd.isna(r['std_test_top5']) else r['std_test_top5']:.2f}",
    axis=1
)
summary_md["Top10 (mean±std)"] = summary_md.apply(
    lambda r: f"{r['mean_test_top10']:.2f} ± {0.0 if pd.isna(r['std_test_top10']) else r['std_test_top10']:.2f}",
    axis=1
)

readme_table = summary_md[
    [
        "model_dim",
        "num_heads",
        "num_layers",
        "ff_multiplier",
        "dropout",
        "lr",
        "pool",
        "runs",
        "Top1 (mean±std)",
        "Top5 (mean±std)",
        "Top10 (mean±std)",
        "best_single_run_top1",
    ]
].head(topk)

display(readme_table)

markdown_text = readme_table.to_markdown(index=False)
print(markdown_text)

,model_dim,num_heads,num_layers,ff_multiplier,dropout,lr,pool,runs,Top1 (mean±std),Top5 (mean±std),Top10 (mean±std),best_single_run_top1


| model_dim   | num_heads   | num_layers   | ff_multiplier   | dropout   | lr   | pool   | runs   | Top1 (mean±std)   | Top5 (mean±std)   | Top10 (mean±std)   | best_single_run_top1   |
|-------------|-------------|--------------|-----------------|-----------|------|--------|--------|-------------------|-------------------|--------------------|------------------------|


In [14]:
import csv
from pathlib import Path
import pandas as pd
import math

raw_csv = ROOT / "results/task3_cislr/temporal_transformer_encoder_gridsearch_results_with_seeds.csv"
clean_csv = ROOT / "results/task3_cislr/temporal_transformer_encoder_gridsearch_results_clean.csv"

expected_cols = [
    "exp_name",
    "input_dim",
    "model_dim",
    "output_dim",
    "num_heads",
    "ff_multiplier",
    "dropout",
    "lr",
    "weight_decay",
    "batch_size",
    "temperature",
    "num_layers",
    "pool",
    "best_val_top1",
    "val_top1",
    "val_top5",
    "val_top10",
    "test_top1",
    "test_top5",
    "test_top10",
    "seed",
    "error",
]

success_cols_21 = [
    "exp_name",
    "input_dim",
    "model_dim",
    "output_dim",
    "num_heads",
    "ff_multiplier",
    "dropout",
    "lr",
    "weight_decay",
    "batch_size",
    "temperature",
    "num_layers",
    "pool",
    "best_val_top1",
    "val_top1",
    "val_top5",
    "val_top10",
    "test_top1",
    "test_top5",
    "test_top10",
    "seed",
]

error_cols_15 = [
    "exp_name",
    "input_dim",
    "model_dim",
    "output_dim",
    "num_heads",
    "ff_multiplier",
    "dropout",
    "lr",
    "weight_decay",
    "batch_size",
    "temperature",
    "num_layers",
    "pool",
    "seed",
    "error",
]

numeric_cols = [
    "input_dim",
    "model_dim",
    "output_dim",
    "num_heads",
    "ff_multiplier",
    "dropout",
    "lr",
    "weight_decay",
    "batch_size",
    "temperature",
    "num_layers",
    "best_val_top1",
    "val_top1",
    "val_top5",
    "val_top10",
    "test_top1",
    "test_top5",
    "test_top10",
    "seed",
]

def to_num(x):
    if x is None:
        return None
    x = str(x).strip()
    if x == "" or x.lower() == "nan":
        return None
    try:
        return float(x)
    except Exception:
        return None

def normalize_error(x):
    if x is None:
        return None
    x = str(x).strip()
    if x == "" or x.lower() == "nan":
        return None
    return x

rows = []
bad_rows = []

with open(raw_csv, "r", encoding="utf-8", newline="") as f:
    reader = csv.reader(f)
    header = next(reader, None)

    for line_idx, row in enumerate(reader, start=2):
        if not row or all(str(x).strip() == "" for x in row):
            continue

        rec = None

        # 21-column successful row
        if len(row) == 21:
            rec = dict(zip(success_cols_21, row))
            rec["error"] = None

        # 22-column row: likely extra leading index
        elif len(row) == 22:
            rec = dict(zip(expected_cols, row[1:]))

        # 15-column early error row
        elif len(row) == 15:
            tmp = dict(zip(error_cols_15, row))
            rec = {col: None for col in expected_cols}
            rec.update(tmp)

        else:
            bad_rows.append({"line_idx": line_idx, "len": len(row), "raw": row})
            continue

        for col in numeric_cols:
            rec[col] = to_num(rec.get(col))

        rec["pool"] = None if rec.get("pool") in ("", None) else str(rec["pool"]).strip()
        rec["exp_name"] = None if rec.get("exp_name") in ("", None) else str(rec["exp_name"]).strip()
        rec["error"] = normalize_error(rec.get("error"))

        for col in ["input_dim", "model_dim", "output_dim", "num_heads", "ff_multiplier", "batch_size", "num_layers", "seed"]:
            if rec[col] is not None and not math.isnan(rec[col]):
                rec[col] = int(rec[col])

        rows.append(rec)

df_clean = pd.DataFrame(rows, columns=expected_cols).drop_duplicates()
df_clean.to_csv(clean_csv, index=False)

print("Clean rows:", len(df_clean))
print("Bad/unparsed rows:", len(bad_rows))
if bad_rows:
    print("Example bad row:", bad_rows[0])

display(df_clean.head())

Clean rows: 194
Bad/unparsed rows: 0


,exp_name,input_dim,model_dim,output_dim,num_heads,ff_multiplier,dropout,lr,weight_decay,batch_size,...,pool,best_val_top1,val_top1,val_top5,val_top10,test_top1,test_top5,test_top10,seed,error
0,temporal_transformer_encoder_dim256_o256_heads...,1024,256,256,4,2,0.0,0.00020,0.0001,32,...,mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1990,'TemporalTransformerEncoder' object has no att...
2,temporal_transformer_encoder_dim256_o256_heads...,1024,256,256,4,2,0.0,0.00020,0.0001,32,...,mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1990,masked_pool() got an unexpected keyword argume...
3,temporal_transformer_encoder_dim256_o256_heads...,1024,256,256,4,2,0.0,0.00020,0.0001,32,...,mean,15.217391,15.217391,20.652174,21.739130,19.124726,24.507659,26.958425,1990,None
4,temporal_transformer_encoder_dim256_o256_heads...,1024,256,256,4,2,0.0,0.00020,0.0001,32,...,mean,15.217391,15.217391,18.478261,21.739130,18.643326,23.150985,25.601751,1990,None
5,temporal_transformer_encoder_dim256_o256_heads...,1024,256,256,4,2,0.0,0.00001,0.0001,32,...,mean,10.869565,10.869565,16.304348,17.391304,16.192560,19.956236,22.013129,1990,None


In [15]:
import pandas as pd

summary_csv = ROOT / "results/task3_cislr/temporal_transformer_gridsearch_summary.csv"

df = pd.read_csv(clean_csv)
df_ok = df[df["error"].isna()].copy()

group_cols = [
    "input_dim",
    "model_dim",
    "output_dim",
    "num_heads",
    "ff_multiplier",
    "dropout",
    "lr",
    "weight_decay",
    "batch_size",
    "temperature",
    "num_layers",
    "pool",
]

summary = (
    df_ok.groupby(group_cols, dropna=False)
    .agg(
        runs=("seed", "count"),
        mean_val_top1=("val_top1", "mean"),
        std_val_top1=("val_top1", "std"),
        mean_test_top1=("test_top1", "mean"),
        std_test_top1=("test_top1", "std"),
        mean_test_top5=("test_top5", "mean"),
        std_test_top5=("test_top5", "std"),
        mean_test_top10=("test_top10", "mean"),
        std_test_top10=("test_top10", "std"),
        best_single_run_top1=("test_top1", "max"),
    )
    .reset_index()
    .sort_values(["mean_test_top1", "mean_test_top5", "mean_test_top10"], ascending=False)
)

summary.to_csv(summary_csv, index=False)

print("Successful runs:", len(df_ok))
print("Unique configs:", len(summary))
display(summary.head(15))

Successful runs: 192
Unique configs: 64


,input_dim,model_dim,output_dim,num_heads,ff_multiplier,dropout,lr,weight_decay,batch_size,temperature,...,runs,mean_val_top1,std_val_top1,mean_test_top1,std_test_top1,mean_test_top5,std_test_top5,mean_test_top10,std_test_top10,best_single_run_top1
18,1024,256,256,8,2,0.0,0.0002,0.0001,32,0.1,...,3,15.942029,2.262680,19.270605,0.291393,24.682713,0.401101,26.900073,0.482063,19.518600
58,1024,512,256,8,4,0.0,0.0002,0.0001,32,0.1,...,3,14.855072,1.255109,19.066375,0.382359,24.011670,0.297893,26.404085,0.133700,19.387309
50,1024,512,256,8,2,0.0,0.0002,0.0001,32,0.1,...,3,15.217391,1.086957,18.949672,0.788961,24.434719,0.248851,26.666667,0.394683,19.824945
34,1024,512,256,4,2,0.0,0.0002,0.0001,32,0.1,...,3,14.492754,1.660354,18.876732,0.463840,24.099198,0.577835,26.725018,0.527590,19.299781
2,1024,256,256,4,2,0.0,0.0002,0.0001,32,0.1,...,3,14.130435,1.086957,18.862144,0.904332,24.449307,0.310486,26.841721,0.322587,19.606127
42,1024,512,256,4,4,0.0,0.0002,0.0001,32,0.1,...,3,15.217391,1.086957,18.570387,0.413638,23.574034,0.597392,26.462436,0.248851,19.037199
54,1024,512,256,8,2,0.1,0.0002,0.0001,32,0.1,...,3,15.217391,1.086957,18.439096,0.291393,22.946754,0.091101,25.222465,0.484046,18.687090
6,1024,256,256,4,2,0.1,0.0002,0.0001,32,0.1,...,3,13.768116,0.627555,18.205689,0.758009,23.282276,0.190761,25.922684,0.297893,18.643326
22,1024,256,256,8,2,0.1,0.0002,0.0001,32,0.1,...,3,15.942029,1.255109,18.074398,0.916951,23.267688,0.692426,25.762217,0.291393,19.080963
26,1024,256,256,8,4,0.0,0.0002,0.0001,32,0.1,...,3,13.405797,2.262680,17.972283,2.095331,22.946754,2.629222,25.543399,2.651347,19.649891


In [17]:
import pandas as pd

summary_csv = ROOT / "results/task3_cislr/temporal_transformer_gridsearch_summary.csv"

df = pd.read_csv(clean_csv)
df_ok = df[df["error"].isna()].copy()

group_cols = [
    "input_dim",
    "model_dim",
    "output_dim",
    "num_heads",
    "ff_multiplier",
    "dropout",
    "lr",
    "weight_decay",
    "batch_size",
    "temperature",
    "num_layers",
    "pool",
]

summary = (
    df_ok.groupby(group_cols, dropna=False)
    .agg(
        runs=("seed", "count"),
        mean_val_top1=("val_top1", "mean"),
        std_val_top1=("val_top1", "std"),
        mean_test_top1=("test_top1", "mean"),
        std_test_top1=("test_top1", "std"),
        mean_test_top5=("test_top5", "mean"),
        std_test_top5=("test_top5", "std"),
        mean_test_top10=("test_top10", "mean"),
        std_test_top10=("test_top10", "std"),
        best_single_run_top1=("test_top1", "max"),
    )
    .reset_index()
    .sort_values(["mean_test_top1", "mean_test_top5", "mean_test_top10"], ascending=False)
)

summary.to_csv(summary_csv, index=False)

print("Successful runs:", len(df_ok))
print("Unique configs:", len(summary))
display(summary)

Successful runs: 192
Unique configs: 64


,input_dim,model_dim,output_dim,num_heads,ff_multiplier,dropout,lr,weight_decay,batch_size,temperature,...,runs,mean_val_top1,std_val_top1,mean_test_top1,std_test_top1,mean_test_top5,std_test_top5,mean_test_top10,std_test_top10,best_single_run_top1
18,1024,256,256,8,2,0.0,0.0002,0.0001,32,0.1,...,3,15.942029,2.262680,19.270605,0.291393,24.682713,0.401101,26.900073,0.482063,19.518600
58,1024,512,256,8,4,0.0,0.0002,0.0001,32,0.1,...,3,14.855072,1.255109,19.066375,0.382359,24.011670,0.297893,26.404085,0.133700,19.387309
50,1024,512,256,8,2,0.0,0.0002,0.0001,32,0.1,...,3,15.217391,1.086957,18.949672,0.788961,24.434719,0.248851,26.666667,0.394683,19.824945
34,1024,512,256,4,2,0.0,0.0002,0.0001,32,0.1,...,3,14.492754,1.660354,18.876732,0.463840,24.099198,0.577835,26.725018,0.527590,19.299781
2,1024,256,256,4,2,0.0,0.0002,0.0001,32,0.1,...,3,14.130435,1.086957,18.862144,0.904332,24.449307,0.310486,26.841721,0.322587,19.606127
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35,1024,512,256,4,2,0.0,0.0002,0.0001,32,0.1,...,3,13.405797,1.660354,15.609044,1.209916,20.058352,2.168106,22.669584,2.423654,16.455142
55,1024,512,256,8,2,0.1,0.0002,0.0001,32,0.1,...,3,11.956522,1.882664,15.609044,1.572044,19.562363,2.307056,21.692195,2.655678,17.024070
43,1024,512,256,4,4,0.0,0.0002,0.0001,32,0.1,...,3,11.231884,1.660354,14.748359,1.608576,18.628738,2.275851,20.685631,2.433643,16.542670
59,1024,512,256,8,4,0.0,0.0002,0.0001,32,0.1,...,3,12.681159,1.255109,14.573304,0.947764,18.832969,1.741956,21.604668,1.929742,15.448578
